# Этап R.1: интерпретатор находит адрес числа подлежащего в незнакомых моделях Pythia

R.0 показал, что адрес есть: R² подмены у DAS 0,839 на последнем слое и 0,559 в точке прихода. R.1 проверяет
интерпретатор: может ли одна обученная модель по сигналам прямого и обратного прохода найти этот адрес в модели
Pythia, которую она не видела, без единой контрфактической пары с неё. Перекрёстное разбиение по seeds: три группы
по три, каждую проверяет интерпретатор, обученный на двух других. Протокол целиком записан в начале кода (ячейка 2.12).

**Этот ноутбук:** протокол зафиксирован, основной прогон.

| раздел | что делает | время на T4 |
|---|---|---|
| 1 | среда и папка на Google Диске | 1 мин |
| 2 | код (два файла с проверкой sha256) и результат R.0 с Диска | 1 мин |
| 3 | пробный запуск на pythia-14m | 2–3 мин |
| 4 | пилот: 10 пилотных моделей, выбор старта и числа эпох | около 1 ч |
| 5 | основной прогон: 36 моделей, 9 интерпретаторов, итог | около 1–1,5 ч |
| 6 | описательно: 410m, затем кривая данных DAS | около 0,5 ч и 1–1,5 ч |
| 7 | сводка и скачивание | — |

**Как запускать.** Среда выполнения → Сменить среду выполнения → **T4 GPU**. Затем Среда выполнения →
**Выполнить все**: выполнятся разделы, у которых включены галочки.

Если сессия оборвётся: снова выполнить разделы 1 и 2, затем прерванный раздел. Обученные интерпретаторы, 410m и
кривая данных сохраняются по одной модели на Диске и берутся готовыми.

Код собран из репозитория: `scripts/pythia_address.py` (sha256 `6c0606476a6e0c1f…`) и `scripts/pythia_interpreter.py` (`6923d17a47957529…`). Результат R.0
(`pythia_address.json`, sha256 `8c802368371d2211…`) берётся с Диска, из папки прогона R.0.

## 1. Среда и папка

In [ ]:
#@title 1.1 Видеокарта и версии
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import sys, torch, transformers, numpy
print("python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA", torch.version.cuda,
      "| transformers", transformers.__version__, "| numpy", numpy.__version__)
assert torch.cuda.is_available(), "нет видеокарты: Среда выполнения -> Сменить среду выполнения -> T4 GPU"


In [ ]:
#@title 1.2 Папка на Google Диске (результаты переживут обрыв сессии)
FOLDER = "/content/drive/MyDrive/R1"  #@param {type:"string"}
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(FOLDER, exist_ok=True)
%cd {FOLDER}


## 2. Код этапа R.1

Два файла исследовательского репозитория целиком встроены в ноутбук: код этапа R.0 (его модели, пары и формулы
использует R.1) и код этапа R.1. Чтобы посмотреть код, нажмите «Показать код». Ячейка сборки записывает файлы и
проверяет sha256: если хоть один символ изменён, она остановится, и прогон будет недействителен. Последняя ячейка
раздела берёт с Диска результат R.0, который воспроизводит контроль 2. **Ничего в этих ячейках не менять.**

In [ ]:
#@title 2.0 Подготовка
PARTS = {}


In [ ]:
#@title 2.1 R.0: Протокол этапа (docstring) и импорты
PARTS[('pythia_address.py', 0)] = r'''#!/usr/bin/env python
"""Stage R.0: does the number of the subject have an address inside pretrained language models?

PROTOCOL. Committed to the repository before the run, once.
---------------------------------------------------------------------------
Role.       Stage 2.0 repeated on models trained by other people: a check of the
            instrument, not of the interpreter. DAS (Geiger, Wu, Potts, Icard and Goodman,
            CLeaR 2024) gives the reference address inside one model. The interpreter is
            tested in stage R.1, under its own protocol, after this result.
            Closest prior work: CausalGym (Arora, Jurafsky and Potts, ACL 2024) ran DAS
            and other methods on Pythia models, including subject-verb agreement
            (agr_sv_num_subj-relc, agr_sv_num_obj-relc, agr_sv_num_pp). They used the
            residual stream after a layer at the last token of a region, one-dimensional
            interventions and a log odds-ratio. Their main text does not map where
            agreement sits, so the layer here is chosen by the pilot below. The metric here
            is stricter: interchange R^2 against the model's own counterfactual output.
Models.     PolyPythias (van der Wal et al., ICLR 2025): EleutherAI/pythia-{size}-seed{1..9},
            sizes 14m, 31m, 70m and 160m. The revisions are pinned in REVISIONS below. Each
            seed changes the initialisation and the batch composition; the corpus (the
            standard Pile) and the hyper-parameters are shared.
            Pilot only, never in the check: seed 0 of every size (the original
            EleutherAI/pythia-{size}) and pythia-160m-{data,weight}-seed{1,2,3}.
            pythia-410m: descriptive only, and only if the time measurement allows it (see
            'Precision').
Task.       Subject-verb number agreement. Three templates over fixed token positions, every
            word one token of the GPT-NeoX tokenizer (checked by the script):
              rc_obj  "The {mod} {head} that the {people} {verb}"   (section 3.7, template 1)
              rc_subj "The {mod} {head} that {verb} the {people}"   (section 3.7, template 2)
              pp      "The {mod} {head} {prep} the {noun}"          (as agr_sv_num_pp)
            The verb of the relative clause agrees with its own subject. B(x) is
            logit(" are") - logit(" is") at the last position.
Variables.  SUBJ, the number of the subject: carriers head (and in rc_subj the clause's
            verb, which agrees with it). ATTR, the number of the attractor: people / noun
            (and in rc_obj the clause's verb). At the last token ATTR is carried by the last
            word itself in every template; its address there is close to the input, so it
            is read separately and its result says so.
Pairs.      Minimal pairs: x_cf is the base sentence with the variable's number taken from
            the source, the base's words kept (the agreeing verb follows). Base and source
            are drawn independently from the same template, the template uniformly. Per
            variable: 4096 training, 1024 validation and 2048 test pairs, numpy
            default_rng([PAIR_STREAM, variable, split]); the same sentences for every model.
            A pair whose base and source have the same number is not informative.
Points.     The residual stream at index r (r = 0 the embeddings, r = j after block j) at the
            last position, k = 1 (the variable is binary). Two main points per size:
              last     r* = the layer chosen by the rule written before the pilot;
              arrival  the first layer whose mean pilot DAS R^2 is >= 0.50: where the number
                       of the subject first arrives at the last token. This criterion was
                       chosen AFTER the pilot, on the pilot models only (seed 0 and the six
                       160m models); the check seeds 1-9 were not used for it.
            Both, from 'Pilot': last 14m 6, 31m 6, 70m 6, 160m 12, 410m 24; arrival 14m 6,
            31m 5, 70m 5, 160m 7, 410m 14. In 14m the two points coincide (decided before
            the run): its arrival values are its last-layer values, not a second fit.
            The subject's own position is a descriptive diagnostic at r = 1. The draft had it
            at the same r as the main point; at r* = the last layer the subject position has no
            path to the output, so every reading there is identically zero (a test shows it).
Metric.     Interchange R^2 = 1 - sum (B(h') - B(x_cf))^2 / sum (B(x_cf) - B(x_b))^2 over
            the test pairs, h' = h_b + ((h_s - h_b) . q) q, as in stage 2.0. A (model,
            variable) with fewer than 50 informative test pairs (|B(x_cf) - B(x_b)| >= 0.10
            sigma_B, sigma_B the standard deviation of B over 256 probe sentences) is left
            out of every mean.
DAS.        Stage 2.0's recipe: q = w / |w|; loss mean(((B(h') - B(x_cf)) / sigma_B)^2);
            Adam, lr 1e-2, 500 steps, batch 256, 3 restarts, the restart with the best
            validation R^2. The result is the test R^2.
Formulas with zero pairs, k = 1, from 4096 sentences of the model's own distribution
            (numpy default_rng([FORMULA_STREAM])), one forward and one backward pass each:
            without labels (the interpreter of R.1 gets the same signals):
              pca          the leading principal component of the residuals;
              first_order  the mean over sentences of J . de, the derivative of the
                           residual along de = e(plural form) - e(singular form) at the
                           variable's carriers (a Jacobian-vector product, label-free:
                           the same direction for every sentence);
              gradient     the mean of dB/dh;
            WITH LABELS of the variable's value on the checked model, which the interpreter
            of R.1 does not get (author's decision, 2026-10-08):
              diffmean     mean residual of plural minus singular sentences;
              lda          (Sigma_w + lambda I)^-1 (mu_1 - mu_0), lambda = 1e-3 tr(Sigma_w)/d;
              probe        L2 logistic regression (1e-3) on residuals scaled by one number;
            and random: the mean over 10 random unit vectors
            (numpy default_rng([RANDOM_STREAM, size, seed, variable])).
Unit.       The seed (1..9). A unit's value is the mean over the four sizes 14m-160m of the
            model's test R^2. With 9 units the smallest two-sided sign-flip p is
            2/512 = 0.0039, so a Holm family of m comparisons can reject only if m <= 12;
            per-size results are descriptive for that reason.
Read-out (a), existence. Family: DAS vs random for SUBJ at the last layer, SUBJ at the
            arrival layer and ATTR at the last layer; exact two-sided sign-flip test over the
            9 units, Holm over these 3. Per (variable, point): CONFIRMED if the comparison
            rejects with a positive difference AND the mean DAS R^2 over the units is
            >= 0.50; NOT CONFIRMED otherwise.
Read-out (b), is the address analytic. Per point, DAS vs each of the six formulas for
            SUBJ, Holm over the six of that point; the two points are reported separately.
            Per formula, as in the check of stage 2.0: ANALYTIC if the 95% bootstrap
            interval of the mean difference DAS - formula lies inside [-0.05, +0.05]; DAS
            HIGHER otherwise if the comparison rejects with a positive difference; FORMULA
            HIGHER if it rejects with a negative one; UNDETERMINED otherwise. Per point:
            ADDRESS ANALYTIC if any formula is ANALYTIC or FORMULA HIGHER; DAS HIGHER THAN
            EVERY FORMULA if all six are DAS HIGHER; MIXED otherwise. The labelled formulas
            are marked. Whatever (b) says, R.1 is run (author's decision); an analytic address
            is reported as a result of its own.
Stop rule.  R.1 is built at each point where (a) is CONFIRMED for SUBJ. If at neither,
            neither R.1 nor part 3 on real models is built.
For R.1, decided by the author on 2026-10-08 and recorded here before R.0's run:
            both points enter R.1's main rule. Per point: (a) the interpreter's R^2 is
            significantly above the strongest formula WITHOUT labels; (b) it is not worse
            than the strongest formula WITH labels by more than 0.05 (TOST). Family: 2 points
            x 2 conditions = 4 comparisons, Holm; each point reported separately (2 if R.0
            confirms one point only). The strongest formulas, chosen on the pilot (see
            'Pilot'): last layer gradient / probe; arrival layer first_order / diffmean.
            R.1's own protocol is written after this result.
Descriptive, outside the families: every size; every template; decision IIA; the
            subject-position diagnostic at r = 1; ATTR's formulas; 410m; the pilot models.
Statistics. 95% percentile bootstrap intervals over the units (10 000 resamples, numpy
            default_rng(0)).
Controls.   On every model, before any fitting; a failure stops that model's run:
            (1) this file's forward pass equals the transformers forward;
            (2) continuing from the cached residual reproduces B(x);
            (3) replacing the whole residual vector at (r, p) by the source's equals the
                same replacement made inside a full forward pass;
            (1)-(3) are computed in float64 (the weights go fp32 -> fp64 -> fp32, which is
                exact), tolerance |difference| <= 1e-8 (1 + |B|). In fp32 both this forward
                pass and the transformers one move B by about 1e-3 on pythia-14m (found on
                the first smoke run, where an fp32 control at 1e-4 stopped); that rounding is
                recorded per model as a descriptive number;
            (4) every minimal pair differs from its base only at the variable's carriers.
Precision.  fp32 for 14m-160m, TF32 off. Determinism flags are set and recorded:
            torch.use_deterministic_algorithms(True, warn_only=True),
            CUBLAS_WORKSPACE_CONFIG=:4096:8, cudnn.deterministic, no cudnn benchmark.
            410m: --time measures a DAS step on the Colab GPU. Rule written before the
            measurement: 410m is run, descriptively and with fp16 autocast, if the measured
            projection of its DAS time for 9 seeds in fp16 (2 variables and the diagnostic,
            3 restarts x 500 steps each) is at most 3 hours; otherwise it is dropped.
            Measured (results/pythia_address_time.json, Tesla T4, one DAS step at batch 256,
            r = L/2): 160m 0.073 s fp32, 0.091 s fp16; 410m 0.217 s fp32, 0.169 s fp16, peak
            GPU memory 6.5 GB. Projection for 410m: 1.91 h in fp16 (2.45 h in fp32). By the
            rule 410m is run, descriptively, in fp16; it may be finished in a second session.
Provenance. Every per-model file and the final file record this script's sha256, torch,
            transformers, numpy and python versions, CUDA, the GPU model, the precision
            mode, the determinism flags and the pinned model revisions. The run is valid only
            if the recorded sha256 equals this file at the protocol commit; `--verify`
            checks that locally before the result is committed.
Amendment 1, 2026-10-09, after the 36 confirmatory models were computed and before any result
            was aggregated. The run stopped at the first descriptive 410m model: under fp16
            autocast the model's outputs (fp16) were written into an fp32 array ("expected scalar
            type Half but found Float"). The time measurement had not met this, because it built
            its pairs outside autocast. Fixed by casting the outputs to fp32, with a test that runs
            the half-precision path. So that the confirmatory part is not recomputed, a per-model
            file of 14m-160m is reused if it was written by the script of the protocol commit
            (PROTOCOL_COMMIT, PROTOCOL_SHA256). The confirmatory result is written to
            pythia_address.json as soon as the 36 models are done, and 410m to its own file
            pythia_address_410m.json. Nothing in the confirmatory computation, the read-outs or
            the families changes (git diff of the two commits); --verify checks the hashes.
Pilot.      On the pilot models only, with a lighter recipe (1024 / 512 pairs, 300 steps,
            1 restart): DAS validation R^2 for SUBJ at the last position at every r (410m every
            second r) and the formulas at every r. Rule written before the pilot: per size, r*
            is the r with the highest mean validation DAS R^2 over that size's pilot models
            (ties: the smaller r); the strongest formula without labels and the strongest with
            labels are the ones with the highest validation R^2 at r*, averaged over the pilot
            models of a size and then over the sizes 14m-160m.
            Run on Colab (Tesla T4, torch 2.11.0+cu130, transformers 5.18.0) with this script
            at commit f808396 (sha256 97e22080...). The first Colab smoke had stopped at
            loading, because transformers 5 renamed the output head; that was fixed in
            f808396 before the pilot. Files: results/pythia_address_pilot.json and
            results/pythia_address_time.json. Every float64 control passed (worst 1.6e-11).
            `--pilot-summary results/pythia_address_pilot.json` reproduces everything below,
            and a test checks that the constants of this file agree with it.
            Mean validation DAS R^2 at the last token by layer r:
              14m  (1 model)  0:.05 1:.05 2:.06 3:.04 4:.05 5:.04 6:.82
              31m  (1)        0:.07 1:.05 2:.07 3:.09 4:.19 5:.53 6:.82
              70m  (1)        0:.08 1:.10 2:.06 3:.20 4:.23 5:.65 6:.86
              160m (7)        0:.05 1:.06 2:.07 3:.05 4:.12 5:.18 6:.24 7:.66 8:.72 9:.70
                              10:.88 11:.89 12:.90
              410m (1)        0:.05 2:.10 4:.06 6:.09 8:.10 10:.17 12:.25 14:.74 16:.76
                              18:.81 20:.82 22:.83 24:.92
            The rule chose the last layer in every size: r* = 6, 6, 6, 12, 24. That is the
            state just before the unembedding. The author then added the arrival layer as a
            second main point (criterion above, chosen after seeing this table).
            Validation R^2 at the two points (DAS | gradient, first_order | probe, diffmean):
              last     14m .82 | .23 .37 | .61 .30     arrival  14m (the same point)
                       31m .82 | .43 .51 | .57 .41              31m r5 .53 | -.11 .27 | .03 .29
                       70m .86 | .64 .47 | .49 .61              70m r5 .65 | .33 .45 | .25 .45
                       160m .90 | .77 .45 | .76 .47             160m r7 .66 | .44 .38 | .45 .47
                       410m .92 | .79 .42 | .83 .70             410m r14 .74 | .62 .18 | .62 .64
            Pooled over 14m-160m, last layer: DAS .851; without labels gradient .517,
            first_order .450, pca -.013; with labels probe .608, diffmean .447, lda .085.
            Arrival layer: DAS .664; without labels first_order .367, gradient .220,
            pca -.002; with labels diffmean .379, probe .337, lda .045.
            Strongest for R.1: last layer gradient / probe (by the rule written before the
            pilot); arrival layer first_order / diffmean (by the same rule applied at the
            arrival layer).
---------------------------------------------------------------------------
Colab (T4):
    !python pythia_address.py --smoke                   # minutes, writes nothing
    !python pythia_address.py --time                    # the 410m time measurement
    !python pythia_address.py --pilot                   # writes pythia_address_pilot.json
    !python pythia_address.py --pilot-summary pythia_address_pilot.json   # the choices from it
    !python pythia_address.py --out R0                  # the run: R0/*.json, then pythia_address.json
Locally, before committing a result:
    python scripts/pythia_address.py --verify pythia_address.json
"""
from __future__ import annotations

import argparse
import hashlib
import json
import os
import platform
import sys
import time
from datetime import datetime, timezone
from itertools import product

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import torch
import torch.nn.functional as F

'''


In [ ]:
#@title 2.2 R.0: Константы, потоки случайных чисел, закреплённые ревизии моделей
PARTS[('pythia_address.py', 1)] = r'''# ------------------------------------------------------------------ constants
SIZES = ("14m", "31m", "70m", "160m")
SIZE_410 = "410m"
SIZE_ID = {"14m": 0, "31m": 1, "70m": 2, "160m": 3, "410m": 4}
CHECK_SEEDS = tuple(range(1, 10))
PILOT_MODELS = tuple(f"pythia-{s}" for s in SIZES + (SIZE_410,)) + tuple(
    f"pythia-160m-{kind}-seed{i}" for kind in ("data", "weight") for i in (1, 2, 3))
VARIABLES = ("SUBJ", "ATTR")
VAR_ID = {"SUBJ": 0, "ATTR": 1}
PAIR_STREAM, FORMULA_STREAM, PROBE_STREAM, RANDOM_STREAM, TORCH_STREAM = 3030, 3031, 3032, 3033, 3034
FULL = dict(n_train=4096, n_val=1024, n_test=2048, steps=500, restarts=3, batch=256, lr=1e-2)
PILOT = dict(n_train=1024, n_val=512, n_test=0, steps=300, restarts=1, batch=256, lr=1e-2)
SMOKE = dict(n_train=64, n_val=32, n_test=32, steps=3, restarts=1, batch=32, lr=1e-2, n_formula=64)
N_FORMULA, N_PROBE, N_RANDOM = 4096, 256, 10
TOL_INFORMATIVE, MIN_INFORMATIVE, BAR, MARGIN = 0.10, 50, 0.50, 0.05
N_BOOT = 10_000
CONTROL_TOL = 1e-8          # |difference| / (1 + |B|), in float64
UNLABELLED = ("pca", "first_order", "gradient")
LABELLED = ("diffmean", "lda", "probe")
FORMULAS = UNLABELLED + LABELLED
# fixed after the pilot, before the commit (see 'Pilot'); pilot_summary() derives them from
# results/pythia_address_pilot.json and a test checks that they agree
R_STAR = {"14m": 6, "31m": 6, "70m": 6, "160m": 12, "410m": 24}     # last layer: rule written before the pilot
R_ARRIVAL = {"14m": 6, "31m": 5, "70m": 5, "160m": 7, "410m": 14}   # arrival: criterion chosen AFTER the pilot
ARRIVAL_BAR = 0.50
R_DIAG = 1                                                          # subject-position diagnostic
STRONGEST = {"last": {"unlabelled": "gradient", "labelled": "probe"},
             "arrival": {"unlabelled": "first_order", "labelled": "diffmean"}}
WITH_410M = True                                                    # descriptive, fp16 (see 'Precision')
# Amendment 1: the 36 confirmatory models were computed by the script of the protocol commit
PROTOCOL_COMMIT = "5a57e39"
PROTOCOL_SHA256 = "c4f4a9dff16f45be682843418b46e5b7ee37a784fd14969fe6cc81aa3ee7f76a"
CONFIRMATORY_FILE, DESCRIPTIVE_410M_FILE = "pythia_address.json", "pythia_address_410m.json"
POINTS = {"last": "SUBJ", "arrival": "SUBJ_arrival"}

REVISIONS = {
    # pinned from the Hugging Face API on 2026-10-08; main = step143000
    "pythia-14m": {"sha": "cf967c0a9a04383db6f7b1108d86b2962634b4ac", "weights": "model.safetensors", "bytes": 28143920},
    "pythia-14m-seed1": {"sha": "0c929f682ef6f3c6582c3f0518713b53117225e2", "weights": "pytorch_model.bin", "bytes": 53331147},
    "pythia-14m-seed2": {"sha": "d34e2feab87a7f8631f7335aac4d85aafbe47663", "weights": "pytorch_model.bin", "bytes": 53331147},
    "pythia-14m-seed3": {"sha": "296092457f9e92450383421fc13c7601cdebba76", "weights": "pytorch_model.bin", "bytes": 53331147},
    "pythia-14m-seed4": {"sha": "c1d2ccdcd54c661e5536c41e185fd97698e0f38d", "weights": "pytorch_model.bin", "bytes": 53331147},
    "pythia-14m-seed5": {"sha": "b86476f2829441a386c32f330d45e06c79699db5", "weights": "pytorch_model.bin", "bytes": 53331592},
    "pythia-14m-seed6": {"sha": "cb8e821ea1e0237339e2ea59cf2666cdf2e9b73b", "weights": "pytorch_model.bin", "bytes": 53331592},
    "pythia-14m-seed7": {"sha": "7bafc17291b96a924536134ced88a045e5152a51", "weights": "pytorch_model.bin", "bytes": 53331592},
    "pythia-14m-seed8": {"sha": "96bed32adeb238865e691de2168a138a513533fb", "weights": "pytorch_model.bin", "bytes": 53331592},
    "pythia-14m-seed9": {"sha": "50e806022dda859254768ba4ceae24788b49aee0", "weights": "pytorch_model.bin", "bytes": 53331592},
    "pythia-31m": {"sha": "e556ace21b489575e94e9d50b6dad2fcc7419679", "weights": "model.safetensors", "bytes": 60997960},
    "pythia-31m-seed1": {"sha": "9d2c5119aa939b5abeef3157c94122fa69e8466a", "weights": "pytorch_model.bin", "bytes": 86185736},
    "pythia-31m-seed2": {"sha": "7a2c0cfe7793936ce1596d1f5f14d00d1e29e7e5", "weights": "pytorch_model.bin", "bytes": 86185736},
    "pythia-31m-seed3": {"sha": "007780274aabaef059324dfb91929e512fbe29ad", "weights": "pytorch_model.bin", "bytes": 86185736},
    "pythia-31m-seed4": {"sha": "500f4c7ccfc248bcff33e6a9ede92af0f77268c4", "weights": "pytorch_model.bin", "bytes": 86185736},
    "pythia-31m-seed5": {"sha": "057af9a8057a13713396963a38f272574f38ff83", "weights": "pytorch_model.bin", "bytes": 86185736},
    "pythia-31m-seed6": {"sha": "285752bb715b22eb0a423c34f3d85799f0a491fd", "weights": "pytorch_model.bin", "bytes": 86185736},
    "pythia-31m-seed7": {"sha": "4baadf03d6cdccbc04bc90a438676bca1e24e5a5", "weights": "pytorch_model.bin", "bytes": 86185736},
    "pythia-31m-seed8": {"sha": "241aa615964e20fb411aea451a241c6d4fbab683", "weights": "pytorch_model.bin", "bytes": 86185736},
    "pythia-31m-seed9": {"sha": "cf8491770336a746113d26ebd41961cbb83cd911", "weights": "pytorch_model.bin", "bytes": 86185736},
    "pythia-70m": {"sha": "a39f36b100fe8a5377810d56c3f4789b9c53ac42", "weights": "model.safetensors", "bytes": 166029852},
    "pythia-70m-seed1": {"sha": "b92214180642ad85f91ccbfffcbf118ebcf2529f", "weights": "pytorch_model.bin", "bytes": 166049099},
    "pythia-70m-seed2": {"sha": "9df96128bfe65ea5e7f1543f0c8ee707932df286", "weights": "pytorch_model.bin", "bytes": 166049099},
    "pythia-70m-seed3": {"sha": "3b039e52757be58fe0f769bec63cf50254853f88", "weights": "pytorch_model.bin", "bytes": 166049099},
    "pythia-70m-seed4": {"sha": "f525bf61a55437bf9166a77571f8e0c054215600", "weights": "pytorch_model.bin", "bytes": 166049099},
    "pythia-70m-seed5": {"sha": "e27a605ee68762a318956435821ef7ac23c21199", "weights": "pytorch_model.bin", "bytes": 166049099},
    "pythia-70m-seed6": {"sha": "b4e7ddaab555b55d9043c210510795ebb0787be1", "weights": "pytorch_model.bin", "bytes": 166049099},
    "pythia-70m-seed7": {"sha": "17e376beb51f5fbf51a47d1e9368fdaa343fe498", "weights": "pytorch_model.bin", "bytes": 166049099},
    "pythia-70m-seed8": {"sha": "ecb4649b5096fed7e206bea3b8abd22192c0c601", "weights": "pytorch_model.bin", "bytes": 166049099},
    "pythia-70m-seed9": {"sha": "373172e086d06a8a9a15816db498d099246f1409", "weights": "pytorch_model.bin", "bytes": 166049099},
    "pythia-160m": {"sha": "50f5173d932e8e61f858120bcb800b97af589f46", "weights": "model.safetensors", "bytes": 374998696},
    "pythia-160m-seed1": {"sha": "5ed55222b3092e69edcfa5fa646b7aac0b7e03d9", "weights": "model.safetensors", "bytes": 374998692},
    "pythia-160m-seed2": {"sha": "6e9b9f6270d1d624f9d7188c1e8909600c4efae2", "weights": "model.safetensors", "bytes": 374998696},
    "pythia-160m-seed3": {"sha": "a74886a25b5f4ffb3b9a0fa80bd02750f13461bd", "weights": "model.safetensors", "bytes": 374998692},
    "pythia-160m-seed4": {"sha": "c7a7ebcc54c75407bfd8cdea09c5cfa2c7961b4d", "weights": "pytorch_model.bin", "bytes": 375036845},
    "pythia-160m-seed5": {"sha": "7682ee9da4d8d399d3ecc3ad468500b3e99cddf3", "weights": "pytorch_model.bin", "bytes": 375036845},
    "pythia-160m-seed6": {"sha": "78e11a97c628c5716c07a277de902f76b99695a7", "weights": "pytorch_model.bin", "bytes": 375036845},
    "pythia-160m-seed7": {"sha": "11b513c3f630f59d0d61cfc0163fc05426f30c07", "weights": "pytorch_model.bin", "bytes": 375036845},
    "pythia-160m-seed8": {"sha": "2149b1efc141db551191f87f778d1f6979ded3e9", "weights": "pytorch_model.bin", "bytes": 375036845},
    "pythia-160m-seed9": {"sha": "08020a8fabd0f0b44d45db35910b9c312b9ade3a", "weights": "pytorch_model.bin", "bytes": 375036845},
    "pythia-410m": {"sha": "9879c9b5f8bea9051dcb0e68dff21493d67e9d4f", "weights": "model.safetensors", "bytes": 911373632},
    "pythia-410m-seed1": {"sha": "33803c4f5a1e9a4bece59e52f17bb8755add33e1", "weights": "pytorch_model.bin", "bytes": 911449213},
    "pythia-410m-seed2": {"sha": "da2549c8d3e03e6e927594b9060073abdb94e6ba", "weights": "pytorch_model.bin", "bytes": 911449213},
    "pythia-410m-seed3": {"sha": "1dd84c3d638fd5b6e3c77815ca2a26f1cdf1a307", "weights": "pytorch_model.bin", "bytes": 911449213},
    "pythia-410m-seed4": {"sha": "57c8ecb67375772335b779545246e2f3a4aed69b", "weights": "pytorch_model.bin", "bytes": 911449213},
    "pythia-410m-seed5": {"sha": "8f2c179eee40284065226c82b9fa0d8106658a6f", "weights": "pytorch_model.bin", "bytes": 911449213},
    "pythia-410m-seed6": {"sha": "6a2463662069b7a4f5e77cf3c222dc60ff6cd3e4", "weights": "pytorch_model.bin", "bytes": 911449213},
    "pythia-410m-seed7": {"sha": "84f51f7862c384d3d3d4095fb3b4d0438022b39e", "weights": "pytorch_model.bin", "bytes": 911449213},
    "pythia-410m-seed8": {"sha": "368c5c53fda3ea210d154fcf6673328fa05da6ca", "weights": "pytorch_model.bin", "bytes": 911449213},
    "pythia-410m-seed9": {"sha": "509afc6f4768f37ebab1705b93a098ecb5d78855", "weights": "pytorch_model.bin", "bytes": 911449213},
    "pythia-160m-data-seed1": {"sha": "dfa32dc674e75495f4046a721de7025d6610109c", "weights": "pytorch_model.bin", "bytes": 375036845},
    "pythia-160m-data-seed2": {"sha": "3b725b0a0f47ca8acdb9bd4d496ac13a52ccab7a", "weights": "pytorch_model.bin", "bytes": 375036845},
    "pythia-160m-data-seed3": {"sha": "c8baaae0f5e9c59fe8e2c54c9e5b4ad4534871ab", "weights": "pytorch_model.bin", "bytes": 375036845},
    "pythia-160m-weight-seed1": {"sha": "fe3e1d29ca2b321c686a0b43d22bf6ed1c3ba6a2", "weights": "pytorch_model.bin", "bytes": 375036845},
    "pythia-160m-weight-seed2": {"sha": "e93e68ff086ab6a5e8b86c129d55e7b5e9c66ea0", "weights": "pytorch_model.bin", "bytes": 375036845},
    "pythia-160m-weight-seed3": {"sha": "0e6423cce79da02daa7975ceb71d6d8d1d77a305", "weights": "pytorch_model.bin", "bytes": 375036845},
}

'''


In [ ]:
#@title 2.3 R.0: Предложения: три шаблона и минимальные пары
PARTS[('pythia_address.py', 2)] = r'''# ------------------------------------------------------------------ the sentences
HEAD = (("nurse", "nurses"), ("doctor", "doctors"), ("author", "authors"), ("pilot", "pilots"),
        ("artist", "artists"), ("farmer", "farmers"), ("teacher", "teachers"), ("driver", "drivers"))
PEOPLE = (("critic", "critics"), ("guard", "guards"), ("student", "students"),
          ("manager", "managers"), ("friend", "friends"))
NOUN = (("window", "windows"), ("table", "tables"), ("door", "doors"), ("garden", "gardens"),
        ("office", "offices"))
VERB = (("likes", "like"), ("knows", "know"), ("helps", "help"), ("meets", "meet"), ("trusts", "trust"))
MODS = ("new", "old", "small", "quiet")
PREPS = ("near", "behind", "beside")
TEMPLATES = ("rc_obj", "rc_subj", "pp")
LENGTH = {"rc_obj": 7, "rc_subj": 7, "pp": 6}
SUBJ_POS = 2
CARRIERS = {"SUBJ": {"rc_obj": (2,), "rc_subj": (2, 4), "pp": (2,)},
            "ATTR": {"rc_obj": (5, 6), "rc_subj": (6,), "pp": (5,)}}
# columns of a sentence record
TPL, MOD, HEADC, SUBJ_PL, ATT, ATTR_PL, VERBC, PREP = range(8)
NUMBER_COL = {"SUBJ": SUBJ_PL, "ATTR": ATTR_PL}


def all_words() -> list[str]:
    out = ["The", "that", "the", "is", "are"] + list(MODS) + list(PREPS)
    for pairs in (HEAD, PEOPLE, NOUN, VERB):
        for a, b in pairs:
            out += [a, b]
    return out


def vocab_ids(tokenizer) -> dict[str, int]:
    """Every word one token; 'The' opens the sentence without a leading space."""
    ids = {}
    for w in all_words():
        enc = tokenizer.encode(w if w == "The" else " " + w)
        if len(enc) != 1:
            raise SystemExit(f"'{w}' is {len(enc)} tokens for this tokenizer. Stopping.")
        ids[w] = enc[0]
    return ids


def sample(n: int, rng: np.random.Generator, tpl: np.ndarray | None = None) -> np.ndarray:
    """n sentence records; numbers of subject and attractor independent and uniform."""
    s = np.zeros((n, 8), dtype=np.int64)
    s[:, TPL] = rng.integers(0, 3, n) if tpl is None else tpl
    s[:, MOD] = rng.integers(0, len(MODS), n)
    s[:, HEADC] = rng.integers(0, len(HEAD), n)
    s[:, SUBJ_PL] = rng.integers(0, 2, n)
    s[:, ATT] = rng.integers(0, 5, n)              # PEOPLE and NOUN both have 5 lemmas
    s[:, ATTR_PL] = rng.integers(0, 2, n)
    s[:, VERBC] = rng.integers(0, len(VERB), n)
    s[:, PREP] = rng.integers(0, len(PREPS), n)
    return s


def words(rec) -> list[str]:
    t = TEMPLATES[rec[TPL]]
    head = HEAD[rec[HEADC]][rec[SUBJ_PL]]
    mod = MODS[rec[MOD]]
    if t == "rc_obj":
        return ["The", mod, head, "that", "the", PEOPLE[rec[ATT]][rec[ATTR_PL]], VERB[rec[VERBC]][rec[ATTR_PL]]]
    if t == "rc_subj":
        return ["The", mod, head, "that", VERB[rec[VERBC]][rec[SUBJ_PL]], "the", PEOPLE[rec[ATT]][rec[ATTR_PL]]]
    return ["The", mod, head, PREPS[rec[PREP]], "the", NOUN[rec[ATT]][rec[ATTR_PL]]]


def token_ids(recs: np.ndarray, vocab: dict[str, int]) -> torch.Tensor:
    """Records of one template -> (n, T) token ids."""
    return torch.tensor([[vocab[w] for w in words(r)] for r in recs], dtype=torch.long)


def tangent_words(rec, variable: str) -> list[tuple[int, str, str]]:
    """(position, singular-side word, plural-side word) at the variable's carriers."""
    t = TEMPLATES[rec[TPL]]
    out = []
    for pos in CARRIERS[variable][t]:
        if pos == 2:
            sg, pl = HEAD[rec[HEADC]]
        elif t == "pp":
            sg, pl = NOUN[rec[ATT]]
        elif (t, pos) in (("rc_obj", 5), ("rc_subj", 6)):
            sg, pl = PEOPLE[rec[ATT]]
        else:                                        # the clause's verb: singular agreement first
            sg, pl = VERB[rec[VERBC]]
        out.append((pos, sg, pl))
    return out


def make_pairs(variable: str, split: int, n: int):
    """(base, source, counterfactual) records of one split; the same for every model."""
    rng = np.random.default_rng([PAIR_STREAM, VAR_ID[variable], split])
    tpl = rng.integers(0, 3, n)
    base = sample(n, rng, tpl)
    src = sample(n, rng, tpl)
    cf = base.copy()
    col = NUMBER_COL[variable]
    cf[:, col] = src[:, col]
    return base, src, cf


def differs_only_at_carriers(base: np.ndarray, cf: np.ndarray, variable: str, vocab) -> bool:
    for b, c in zip(base, cf):
        wb, wc = words(b), words(c)
        diff = {i for i, (x, y) in enumerate(zip(wb, wc)) if x != y}
        if not diff <= set(CARRIERS[variable][TEMPLATES[b[TPL]]]):
            return False
    return True


'''


In [ ]:
#@title 2.4 R.0: Прямой проход GPT-NeoX с кэшем префикса
PARTS[('pythia_address.py', 3)] = r'''# ------------------------------------------------------------------ the model
def rope_params(cfg) -> tuple[float, float]:
    """(base, rotary share) of the rotary embedding: config.rope_parameters in transformers 5,
    rotary_emb_base / rotary_pct in transformers 4."""
    rp = getattr(cfg, "rope_parameters", None)
    if isinstance(rp, dict) and "rope_theta" in rp:
        if rp.get("rope_type", "default") != "default":
            raise SystemExit(f"rope type {rp.get('rope_type')} is not the one this forward pass implements")
        return float(rp["rope_theta"]), float(rp.get("partial_rotary_factor", 1.0))
    if getattr(cfg, "rope_scaling", None):
        raise SystemExit("rope scaling is not implemented in this forward pass")
    pct = getattr(cfg, "rotary_pct", None)
    pct = getattr(cfg, "partial_rotary_factor", 1.0) if pct is None else pct
    base = getattr(cfg, "rotary_emb_base", None)
    base = getattr(cfg, "rope_theta", 10000) if base is None else base
    return float(base), float(pct)


class Neox:
    """GPT-NeoX forward pass written out, so that a residual can be cached and continued.

    Uses the transformers module weights; control (1) checks it against the transformers
    forward pass on every model.
    """

    def __init__(self, hf, are_id: int, is_id: int):
        self.hf = hf
        self.m = hf.gpt_neox
        self.blocks = list(self.m.layers)
        self.L = len(self.blocks)
        cfg = hf.config
        if not getattr(cfg, "use_parallel_residual", True):
            raise SystemExit("this forward pass assumes the parallel residual of Pythia")
        self.H = cfg.num_attention_heads
        self.hs = cfg.hidden_size // self.H
        self.d = cfg.hidden_size
        base, pct = rope_params(cfg)
        self.rot = int(self.hs * pct)
        dev = next(hf.parameters()).device
        self.inv_freq = 1.0 / (base ** (torch.arange(0, self.rot, 2, dtype=torch.float32, device=dev) / self.rot))
        self.are_id, self.is_id = are_id, is_id
        head = hf.get_output_embeddings()           # embed_out in transformers 4, lm_head in 5
        self.w = (head.weight[are_id] - head.weight[is_id]).detach()
        self.bias = 0.0
        if getattr(head, "bias", None) is not None:
            self.bias = float(head.bias[are_id] - head.bias[is_id])
        self.device = dev

    def rope(self, pos: torch.Tensor):
        freqs = pos[:, None].float() * self.inv_freq[None]
        emb = torch.cat([freqs, freqs], -1)
        return emb.cos(), emb.sin()

    @staticmethod
    def _rotate(x, cos, sin, rot):
        xr, xp = x[..., :rot], x[..., rot:]
        half = rot // 2
        rh = torch.cat([-xr[..., half:], xr[..., :half]], -1)
        return torch.cat([xr * cos + rh * sin, xp], -1)

    def block(self, j: int, h: torch.Tensor, pos: torch.Tensor, past=None):
        """One block on the positions `pos` of h (n, t, d); past = (K, V) of the earlier positions."""
        blk = self.blocks[j]
        n, t, _ = h.shape
        x = blk.input_layernorm(h)
        qkv = blk.attention.query_key_value(x).view(n, t, self.H, 3 * self.hs).transpose(1, 2)
        q, k, v = qkv.chunk(3, -1)
        cos, sin = self.rope(pos)
        q = self._rotate(q, cos.to(q.dtype), sin.to(q.dtype), self.rot)
        k = self._rotate(k, cos.to(k.dtype), sin.to(k.dtype), self.rot)
        if past is not None:
            k = torch.cat([past[0], k], 2)
            v = torch.cat([past[1], v], 2)
        scores = (q @ k.transpose(-1, -2)) * self.hs ** -0.5
        kpos = torch.arange(k.shape[2], device=h.device)
        mask = kpos[None, :] > pos[:, None]
        scores = scores.masked_fill(mask, float("-inf"))
        low = scores.dtype in (torch.float16, torch.bfloat16)        # as transformers: softmax in fp32
        probs = torch.softmax(scores.float() if low else scores, -1).to(q.dtype)
        o = (probs @ v).transpose(1, 2).reshape(n, t, self.d)
        a = blk.attention.dense(o)
        m = blk.mlp(blk.post_attention_layernorm(h))
        return h + a + m, (k, v)

    def readout(self, h_last: torch.Tensor) -> torch.Tensor:
        return self.m.final_layer_norm(h_last) @ self.w + self.bias

    def embed(self, ids: torch.Tensor) -> torch.Tensor:
        return self.m.embed_in(ids.to(self.device))

    def forward(self, ids: torch.Tensor, patch=None) -> torch.Tensor:
        """B(x); patch = (r, p, new residual (n, d)) replaces the residual at index r, position p."""
        h = self.embed(ids)
        T = h.shape[1]
        pos = torch.arange(T, device=self.device)
        for j in range(self.L + 1):
            if patch is not None and patch[0] == j:
                h = h.clone()
                h[:, patch[1]] = patch[2]
            if j == self.L:
                break
            h, _ = self.block(j, h, pos)
        return self.readout(h[:, -1])

    def resid(self, h: torch.Tensor, r: int) -> torch.Tensor:
        """The residual stream at index r of every position, from the embeddings h."""
        pos = torch.arange(h.shape[1], device=h.device)
        for j in range(r):
            h, _ = self.block(j, h, pos)
        return h

    def prefix(self, ids: torch.Tensor, r: int, p: int) -> dict:
        """The cache for continuing from (r, p): the residual at index r for positions p..T-1 and
        the keys and values of blocks r..L-1 at positions 0..p-1."""
        with torch.no_grad():
            h = self.embed(ids)
            T = h.shape[1]
            pos = torch.arange(T, device=self.device)
            for j in range(r):
                h, _ = self.block(j, h, pos)
            hr = h[:, p:].clone()
            past = []
            hh = h
            for j in range(r, self.L):
                hh, (k, v) = self.block(j, hh, pos)
                past.append((k[:, :, :p].clone(), v[:, :, :p].clone()))
        return {"h": hr, "past": past, "r": r, "p": p, "T": T}

    def cont(self, cache: dict, new_p: torch.Tensor, idx=None) -> torch.Tensor:
        """B with the residual at (r, p) replaced by new_p (n, d); idx selects rows of the cache."""
        r, p, T = cache["r"], cache["p"], cache["T"]
        h = cache["h"] if idx is None else cache["h"][idx]
        h = torch.cat([new_p[:, None].to(h.dtype), h[:, 1:]], 1)
        pos = torch.arange(p, T, device=h.device)
        for j in range(r, self.L):
            k, v = cache["past"][j - r]
            if idx is not None:
                k, v = k[idx], v[idx]
            h, _ = self.block(j, h, pos, past=(k, v) if p > 0 else None)
        return self.readout(h[:, -1])


def slice_cache(cache: dict, idx) -> dict:
    return {"h": cache["h"][idx], "past": [(k[idx], v[idx]) for k, v in cache["past"]],
            "r": cache["r"], "p": cache["p"], "T": cache["T"]}


'''


In [ ]:
#@title 2.5 R.0: Наборы пар, подмена на последнем токене, DAS
PARTS[('pythia_address.py', 4)] = r'''# ------------------------------------------------------------------ batched helpers over mixed templates
CHUNK = 512


def by_template(recs: np.ndarray) -> dict[int, np.ndarray]:
    return {t: np.flatnonzero(recs[:, TPL] == t) for t in range(3) if (recs[:, TPL] == t).any()}


def outputs(model: Neox, recs: np.ndarray, vocab) -> torch.Tensor:
    out = torch.empty(len(recs), device=model.device)
    with torch.no_grad():
        for t, ix in by_template(recs).items():
            for s in range(0, len(ix), CHUNK):
                part = ix[s:s + CHUNK]
                out[torch.as_tensor(part, device=model.device)] = model.forward(token_ids(recs[part], vocab)).float()
    return out


def point(t: int, where: str) -> int:
    return LENGTH[TEMPLATES[t]] - 1 if where == "last" else SUBJ_POS


class PairSet:
    """One split of minimal pairs on one model, ready for the interchange at (r, where)."""

    def __init__(self, model: Neox, vocab, base, src, cf, r: int, where: str, sigma: float):
        self.model, self.r, self.where = model, r, where
        self.groups = by_template(base)
        self.n = len(base)
        self.y_b = outputs(model, base, vocab)
        self.y_cf = outputs(model, cf, vocab)
        self.sigma = sigma
        self.cache, self.h_s, self.local = {}, {}, np.zeros(self.n, dtype=np.int64)
        for t, ix in self.groups.items():
            p = point(t, where)
            self.cache[t] = model.prefix(token_ids(base[ix], vocab), r, p)
            with torch.no_grad():
                self.h_s[t] = model.resid(model.embed(token_ids(src[ix], vocab)), r)[:, p]
            self.local[ix] = np.arange(len(ix))
        self.order = np.concatenate([self.groups[t] for t in self.groups])
        dev = model.device
        self.yo_cf = self.y_cf[torch.as_tensor(self.order, device=dev)].double().cpu().numpy()
        self.yo_b = self.y_b[torch.as_tensor(self.order, device=dev)].double().cpu().numpy()

    def predict(self, q: torch.Tensor) -> np.ndarray:
        """B(h') for every pair, in the order self.order, for the unit direction q."""
        preds = []
        with torch.no_grad():
            for t, ix in self.groups.items():
                for s in range(0, len(ix), CHUNK):
                    li = torch.arange(s, min(s + CHUNK, len(ix)), device=self.model.device)
                    hb = self.cache[t]["h"][li, 0]
                    hs = self.h_s[t][li]
                    new = hb + ((hs - hb) @ q)[:, None] * q[None]
                    preds.append(self.model.cont(self.cache[t], new, idx=li))
        return torch.cat(preds).double().cpu().numpy()

    def informative(self) -> int:
        return int((np.abs(self.yo_cf - self.yo_b) >= TOL_INFORMATIVE * self.sigma).sum())

    def r2(self, q: torch.Tensor) -> tuple[float, int, float]:
        """(interchange R^2, informative pairs, decision IIA) over the whole split."""
        pr, y_cf, y_b = self.predict(q), self.yo_cf, self.yo_b
        den = float(((y_cf - y_b) ** 2).sum())
        val = float(1.0 - ((pr - y_cf) ** 2).sum() / den) if den > 0 else float("nan")
        flip = np.sign(y_cf) != np.sign(y_b)
        dec = float((np.sign(pr[flip]) == np.sign(y_cf[flip])).mean()) if flip.any() else float("nan")
        return val, self.informative(), dec

    def r2_by_template(self, q: torch.Tensor) -> dict[str, float]:
        pr = self.predict(q)
        out, start = {}, 0
        for t, ix in self.groups.items():
            sl = slice(start, start + len(ix))
            start += len(ix)
            y_cf, y_b = self.yo_cf[sl], self.yo_b[sl]
            den = float(((y_cf - y_b) ** 2).sum())
            out[TEMPLATES[t]] = float(1.0 - ((pr[sl] - y_cf) ** 2).sum() / den) if den > 0 else float("nan")
        return out


def seed_of(*parts) -> int:
    return int(np.random.SeedSequence([int(p) for p in parts]).generate_state(1)[0])


def das_steps(train: PairSet, c: dict, g: torch.Generator, steps: int) -> torch.Tensor:
    """Adam on the interchange loss from a random start; returns the unit direction."""
    w = torch.randn(train.model.d, generator=g).to(train.model.device).requires_grad_(True)
    opt = torch.optim.Adam([w], lr=c["lr"])
    for _ in range(steps):
        rows = torch.randint(0, train.n, (min(c["batch"], train.n),), generator=g).numpy()
        q = w / w.norm()
        loss = 0.0
        for t, ix in train.groups.items():
            sel = rows[np.isin(rows, ix)]
            if len(sel) == 0:
                continue
            loc = torch.as_tensor(train.local[sel], device=train.model.device)
            hb = train.cache[t]["h"][loc, 0]
            hs = train.h_s[t][loc]
            new = hb + ((hs - hb) @ q)[:, None] * q[None]
            pred = train.model.cont(train.cache[t], new, idx=loc)
            ref = train.y_cf[torch.as_tensor(sel, device=train.model.device)]
            loss = loss + (((pred - ref) / train.sigma) ** 2).sum()
        loss = loss / len(rows)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return (w / w.norm()).detach()


def fit_das(train: PairSet, val: PairSet, c: dict, seed_parts: tuple) -> tuple[torch.Tensor, float]:
    """Stage 2.0's DAS at k = 1; the restart with the best validation R^2 (ties: the lower index)."""
    best = None
    for rs in range(c["restarts"]):
        g = torch.Generator().manual_seed(seed_of(*seed_parts, rs))
        q = das_steps(train, c, g, c["steps"])
        v = val.r2(q)[0]
        key = (-np.inf if np.isnan(v) else v, -rs)
        if best is None or key > best[0]:
            best = (key, q, v)
    return best[1], best[2]


'''


In [ ]:
#@title 2.6 R.0: Формулы с нулём пар: без меток и с метками
PARTS[('pythia_address.py', 5)] = r'''# ------------------------------------------------------------------ formulas with zero pairs
def formula_inputs(model: Neox, recs: np.ndarray, vocab, variable: str, r: int, where: str) -> dict:
    """Residuals H, gradients G = dB/dh and Jacobian-vector products D along the plural direction
    at (r, where), and the variable's labels, on the model's own sentences."""
    H, G, D, y = [], [], [], []
    emb = model.m.embed_in.weight
    for t, ix in by_template(recs).items():
        p = point(t, where)
        for s in range(0, len(ix), CHUNK):
            part = recs[ix[s:s + CHUNK]]
            ids = token_ids(part, vocab).to(model.device)
            cache = model.prefix(ids, r, p)
            hp = cache["h"][:, 0].clone().requires_grad_(True)
            out = model.cont(cache, hp)
            g = torch.autograd.grad(out.sum(), hp)[0]
            e0 = model.embed(ids).detach()
            tan = torch.zeros_like(e0)
            vid = {w: vocab[w] for w in vocab}
            for n_, rec in enumerate(part):
                for pos, sg, pl in tangent_words(rec, variable):
                    tan[n_, pos] = (emb[vid[pl]] - emb[vid[sg]]).detach()
            fn = lambda e: model.resid(e, r)[:, p]
            hval, dval = torch.func.jvp(fn, (e0,), (tan,))
            H.append(hval.detach().double().cpu())
            D.append(dval.detach().double().cpu())
            G.append(g.detach().double().cpu())
            y.append(torch.as_tensor(part[:, NUMBER_COL[variable]], dtype=torch.float64))
    return {"H": torch.cat(H), "G": torch.cat(G), "D": torch.cat(D), "y": torch.cat(y)}


def _unit(v: torch.Tensor) -> torch.Tensor:
    return v / v.norm().clamp(min=1e-30)


def formulas(fi: dict) -> dict[str, torch.Tensor]:
    H, G, D, y = fi["H"], fi["G"], fi["D"], fi["y"]
    d = H.shape[1]
    Hc = H - H.mean(0)
    cov = Hc.T @ Hc / len(H)
    out = {"pca": _unit(torch.linalg.eigh(cov)[1][:, -1]),
           "first_order": _unit(D.mean(0)),
           "gradient": _unit(G.mean(0))}
    m1, m0 = H[y == 1].mean(0), H[y == 0].mean(0)
    out["diffmean"] = _unit(m1 - m0)
    X1, X0 = H[y == 1] - m1, H[y == 0] - m0
    sw = (X1.T @ X1 + X0.T @ X0) / len(H)
    lam = 1e-3 * torch.trace(sw) / d
    out["lda"] = _unit(torch.linalg.solve(sw + lam * torch.eye(d, dtype=sw.dtype), m1 - m0))
    scale = Hc.pow(2).mean().sqrt().clamp(min=1e-30)
    X = Hc / scale
    w = torch.zeros(d, dtype=X.dtype, requires_grad=True)
    b = torch.zeros(1, dtype=X.dtype, requires_grad=True)
    opt = torch.optim.LBFGS([w, b], max_iter=500, tolerance_grad=1e-10, tolerance_change=1e-12,
                            line_search_fn="strong_wolfe")
    sgn = 2 * y - 1

    def closure():
        opt.zero_grad()
        loss = F.softplus(-sgn * (X @ w + b)).mean() + 0.5e-3 * (w * w).sum()
        loss.backward()
        return loss
    opt.step(closure)
    out["probe"] = _unit(w.detach())
    return out


def random_dirs(d: int, size: str, seed: int, variable: str) -> list[torch.Tensor]:
    rng = np.random.default_rng([RANDOM_STREAM, SIZE_ID[size], seed, VAR_ID[variable]])
    return [_unit(torch.as_tensor(rng.standard_normal(d))) for _ in range(N_RANDOM)]


'''


In [ ]:
#@title 2.7 R.0: Статистика: перестановочный тест, Холм, бутстрэп
PARTS[('pythia_address.py', 6)] = r'''# ------------------------------------------------------------------ statistics (as mint.eval.stats)
def paired_sign_flip(a, b) -> tuple[float, float]:
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    d = d[np.isfinite(d)]
    m = len(d)
    if m == 0:
        return float("nan"), float("nan")
    obs = abs(d.mean())
    signs = np.array(list(product([1, -1], repeat=m)), dtype=float)
    null = np.abs((signs * d).mean(axis=1))
    return float(d.mean()), float((null >= obs - 1e-12).mean())


def holm(pvals: list[float], alpha: float = 0.05) -> list[dict]:
    idx = [i for i, p in enumerate(pvals) if np.isfinite(p)]
    order = sorted(idx, key=lambda i: pvals[i])
    n = len(order)
    out = [{"p": p, "p_adj": float("nan"), "reject": False} for p in pvals]
    running = 0.0
    for rank, i in enumerate(order):
        adj = min(1.0, max(running, (n - rank) * pvals[i]))
        running = adj
        out[i] = {"p": pvals[i], "p_adj": adj, "reject": bool(adj <= alpha)}
    return out


def boot(vals, n: int = N_BOOT) -> list[float]:
    v = np.asarray([x for x in vals if np.isfinite(x)], dtype=float)
    if len(v) == 0:
        return [float("nan"), float("nan")]
    rng = np.random.default_rng(0)
    means = v[rng.integers(0, len(v), (n, len(v)))].mean(1)
    return [float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))]


'''


In [ ]:
#@title 2.8 R.0: Среда, детерминизм, загрузка моделей
PARTS[('pythia_address.py', 7)] = r'''# ------------------------------------------------------------------ environment and models
def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def set_determinism():
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


def environment(precision: dict) -> dict:
    import transformers
    cuda = torch.cuda.is_available()
    return {"script_sha256": sha256_file(__file__), "torch": torch.__version__,
            "transformers": transformers.__version__, "numpy": np.__version__,
            "python": platform.python_version(), "cuda": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0) if cuda else None,
            "colab": "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG")),
            "precision": precision,
            "determinism": {"use_deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
                            "warn_only": torch.is_deterministic_algorithms_warn_only_enabled(),
                            "CUBLAS_WORKSPACE_CONFIG": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
                            "cudnn_deterministic": torch.backends.cudnn.deterministic,
                            "cudnn_benchmark": torch.backends.cudnn.benchmark,
                            "matmul_allow_tf32": torch.backends.cuda.matmul.allow_tf32,
                            "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32}}


def repo(name: str) -> str:
    return f"EleutherAI/{name}"


def check_name(size: str, seed: int) -> str:
    return f"pythia-{size}-seed{seed}"


def load(name: str, device, fp16_autocast: bool = False):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    rev = REVISIONS.get(name, {}).get("sha")
    if rev is None:
        raise SystemExit(f"{name}: no pinned revision. Stopping.")
    tk = AutoTokenizer.from_pretrained(repo(name), revision=rev)
    hf = AutoModelForCausalLM.from_pretrained(repo(name), revision=rev)
    hf = hf.float().to(device).eval()          # fp32 whatever the checkpoint and the library default
    for p in hf.parameters():
        p.requires_grad_(False)
    vocab = vocab_ids(tk)
    return hf, Neox(hf, vocab["are"], vocab["is"]), vocab, rev


'''


In [ ]:
#@title 2.9 R.0: Контроли (в float64)
PARTS[('pythia_address.py', 8)] = r'''# ------------------------------------------------------------------ controls
def controls(hf, model: Neox, vocab, r: int, rng_seed: int = 0) -> dict:
    """The protocol's controls (1)-(3), computed in float64, where they are exact up to rounding;
    and, descriptive, how far fp32 rounding moves B on the same sentences. The weights go
    fp32 -> fp64 -> fp32, which is exact."""
    rng = np.random.default_rng([PROBE_STREAM, 99, rng_seed])
    worst = {"forward": 0.0, "continue": 0.0, "swap": 0.0}
    sets = []
    hf.double()
    try:
        m64 = Neox(hf, model.are_id, model.is_id)
        with torch.no_grad():
            for t in range(3):
                ids = token_ids(sample(64, rng, np.full(64, t)), vocab).to(model.device)
                ids_s = token_ids(sample(64, rng, np.full(64, t)), vocab).to(model.device)
                mine = m64.forward(ids)
                logits = hf(input_ids=ids).logits[:, -1]
                ref = logits[:, vocab["are"]] - logits[:, vocab["is"]]
                worst["forward"] = max(worst["forward"], float(((mine - ref).abs() / (1 + ref.abs())).max()))
                for where in ("last", "subj"):
                    p = point(t, where)
                    cache = m64.prefix(ids, r, p)
                    again = m64.cont(cache, cache["h"][:, 0])
                    worst["continue"] = max(worst["continue"], float(((again - mine).abs() / (1 + mine.abs())).max()))
                    hs = m64.resid(m64.embed(ids_s), r)[:, p]
                    via_cache = m64.cont(cache, hs)
                    via_patch = m64.forward(ids, patch=(r, p, hs))
                    worst["swap"] = max(worst["swap"], float(((via_cache - via_patch).abs() / (1 + via_patch.abs())).max()))
                sets.append((ids, mine))
    finally:
        hf.float()
    with torch.no_grad():
        rounding = max(float(((model.forward(ids).double() - mine).abs() / (1 + mine.abs())).max()) for ids, mine in sets)
    return {"float64": worst, "fp32_rounding": rounding}


def controls_failed(ctl: dict) -> bool:
    return max(ctl["float64"].values()) > CONTROL_TOL


'''


In [ ]:
#@title 2.10 R.0: Расчёт одной модели
PARTS[('pythia_address.py', 9)] = r'''# ------------------------------------------------------------------ one model
def probe_sigma(model: Neox, vocab) -> float:
    recs = sample(N_PROBE, np.random.default_rng([PROBE_STREAM]))
    return float(outputs(model, recs, vocab).std().item())


def pair_sets(model, vocab, variable, r, where, sigma, c, splits=(0, 1, 2)):
    sizes = (c["n_train"], c["n_val"], c["n_test"])
    out = {}
    for split in splits:
        if sizes[split] == 0:
            continue
        base, src, cf = make_pairs(variable, split, sizes[split])
        if not differs_only_at_carriers(base, cf, variable, vocab):
            raise SystemExit("control (4): a minimal pair differs outside the carriers. Stopping.")
        out[split] = PairSet(model, vocab, base, src, cf, r, where, sigma)
    return out


def evaluate_point(model, vocab, size, seed, variable, r, where, sigma, c, test_split=2,
                   with_das=True) -> dict:
    sets = pair_sets(model, vocab, variable, r, where, sigma, c)
    ev = sets[test_split]
    rec = {"r": r, "where": where}
    if with_das:
        q, val = fit_das(sets[0], sets[1], c, (TORCH_STREAM, SIZE_ID[size], seed, VAR_ID[variable], r,
                                               0 if where == "last" else 1))
        rec["das"], rec["n_informative"], rec["das_decision_iia"] = ev.r2(q)
        rec["das_validation"] = val
        rec["das_by_template"] = ev.r2_by_template(q)
    fi = formula_inputs(model, sample(c.get("n_formula", N_FORMULA), np.random.default_rng([FORMULA_STREAM])),
                        vocab, variable, r, where)
    for name, v in formulas(fi).items():
        rec[name] = ev.r2(v.to(model.device, torch.float32))[0]
    rec["random"] = float(np.nanmean([ev.r2(v.to(model.device, torch.float32))[0]
                                      for v in random_dirs(model.d, size, seed, variable)]))
    rec["n_informative"] = ev.informative()
    return rec


def run_model(name: str, size: str, seed: int, r: int, c: dict, device, out_dir: str,
              precision: str = "fp32") -> dict:
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"{name}.json")
    if os.path.exists(path):
        old = json.load(open(path))
        sha = old["environment"]["script_sha256"]
        if sha == sha256_file(__file__) or (size in SIZES and sha == PROTOCOL_SHA256):
            print(f"{name}: already done, kept", flush=True)
            return old
    t0 = time.time()
    hf, model, vocab, rev = load(name, device)
    ctl = controls(hf, model, vocab, r)
    if controls_failed(ctl):
        raise SystemExit(f"{name}: control failed {ctl}. Stopping.")
    sigma = probe_sigma(model, vocab)
    rec = {"model": name, "repo": repo(name), "revision": rev, "size": size, "seed": seed,
           "checkpoint_dtype": str(getattr(hf.config, "dtype", None) or getattr(hf.config, "torch_dtype", None)),
           "sigma": sigma, "controls": ctl,
           "point": {"last": r, "arrival": R_ARRIVAL.get(size, r), "diagnostic_subject_position": R_DIAG}}
    ac = torch.autocast("cuda", dtype=torch.float16) if precision == "fp16" else torch.autocast("cpu", enabled=False)
    with ac:
        for var in VARIABLES:
            rec[var] = evaluate_point(model, vocab, size, seed, var, r, "last", sigma, c)
            print(f"  {name} {var}: DAS {rec[var]['das']:+.3f} " +
                  " ".join(f"{k} {rec[var][k]:+.3f}" for k in FORMULAS + ('random',)), flush=True)
        ra = R_ARRIVAL.get(size, r)
        if ra != r:
            rec["SUBJ_arrival"] = evaluate_point(model, vocab, size, seed, "SUBJ", ra, "last", sigma, c)
            print(f"  {name} SUBJ at arrival r={ra}: DAS {rec['SUBJ_arrival']['das']:+.3f}", flush=True)
        else:
            rec["SUBJ_arrival"] = dict(rec["SUBJ"], same_as_last=True)     # 14m: the two points coincide
        rec["SUBJ_subject_position"] = evaluate_point(model, vocab, size, seed, "SUBJ", R_DIAG, "subj", sigma, c)
    rec["seconds"] = time.time() - t0
    rec["environment"] = environment({"mode": precision, "autocast_fp16": precision == "fp16"})
    with open(path + ".tmp", "w") as f:
        json.dump(rec, f, indent=1)
    os.replace(path + ".tmp", path)
    del hf, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return rec


'''


In [ ]:
#@title 2.11 R.0: Пилот, замер, основной прогон, сводка
PARTS[('pythia_address.py', 10)] = r'''# ------------------------------------------------------------------ pilot, time, smoke, run, verify
def pilot(device, c: dict, out: str, models=PILOT_MODELS, smoke: bool = False,
          parts: str = "pilot_parts") -> dict:
    """The layer sweep on the pilot models; one file per model in `parts`, so that a broken
    session resumes where it stopped (a file is reused only if the script and config match)."""
    res = {}
    os.makedirs(parts, exist_ok=True)
    me = sha256_file(__file__)
    for name in models:
        size = name.split("-")[1]
        path = os.path.join(parts, f"{name}.json")
        if os.path.exists(path):
            old = json.load(open(path))
            if old.get("script_sha256") == me and old.get("config") == c:
                old["profile"] = {int(k): v for k, v in old["profile"].items()}
                res[name] = old
                print(f"  pilot {name}: already done, kept", flush=True)
                continue
        hf, model, vocab, rev = load(name, device)
        sigma = probe_sigma(model, vocab)
        rs = list(range(model.L + 1)) if size != SIZE_410 else list(range(0, model.L + 1, 2))
        if smoke:
            rs = rs[:2]
        prof = {}
        for r in rs:
            ctl = controls(hf, model, vocab, r)
            if controls_failed(ctl):
                raise SystemExit(f"{name} r={r}: control failed {ctl}. Stopping.")
            sets = pair_sets(model, vocab, "SUBJ", r, "last", sigma, c, splits=(0, 1))
            q, val = fit_das(sets[0], sets[1], c, (TORCH_STREAM, 90 + SIZE_ID[size], len(res), 0, r, 0))
            fi = formula_inputs(model, sample(c.get("n_formula", N_FORMULA),
                                              np.random.default_rng([FORMULA_STREAM])), vocab, "SUBJ", r, "last")
            row = {"das": val, "controls": ctl}
            for k, v in formulas(fi).items():
                row[k] = sets[1].r2(v.to(model.device, torch.float32))[0]
            prof[r] = row
            print(f"  pilot {name} r={r}: DAS {val:+.3f} " +
                  " ".join(f"{k} {row[k]:+.3f}" for k in FORMULAS), flush=True)
        res[name] = {"size": size, "revision": rev, "sigma": sigma, "profile": prof,
                     "script_sha256": me, "config": c, "environment": environment({"mode": "fp32"})}
        with open(path + ".tmp", "w") as f:
            json.dump(res[name], f, indent=1)
        os.replace(path + ".tmp", path)
        del hf, model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    # the rule written before the pilot
    choice = {}
    for size in SIZES + (SIZE_410,):
        names = [n for n in res if res[n]["size"] == size]
        if not names:
            continue
        rs = sorted(res[names[0]]["profile"])
        mean = {r: float(np.nanmean([res[n]["profile"][r]["das"] for n in names])) for r in rs}
        best = max(rs, key=lambda r: (mean[r], -r))
        choice[size] = {"r_star": best, "mean_validation_das": mean,
                        "formulas_at_r_star": {k: float(np.nanmean([res[n]["profile"][best][k] for n in names]))
                                               for k in FORMULAS}}
    pooled = {k: float(np.nanmean([choice[s]["formulas_at_r_star"][k] for s in SIZES if s in choice]))
              for k in FORMULAS}
    strongest = {"unlabelled": max(UNLABELLED, key=lambda k: pooled[k]),
                 "labelled": max(LABELLED, key=lambda k: pooled[k])}
    doc = {"pilot": res, "choice": choice, "formulas_pooled_14m_160m": pooled, "strongest": strongest,
           "config": c, "environment": environment({"mode": "fp32"})}
    if out:
        with open(out, "w") as f:
            json.dump(doc, f, indent=1)
    print("r* per size:", {s: v["r_star"] for s, v in choice.items()}, "strongest:", strongest, flush=True)
    return doc


def time_410(device) -> dict:
    """A short measurement of one DAS step at batch 256 for 160m and 410m, fp32 and fp16 autocast."""
    out = {}
    for name in ("pythia-160m", "pythia-410m"):
        hf, model, vocab, rev = load(name, device)
        r = model.L // 2
        sets = pair_sets(model, vocab, "SUBJ", r, "last", 1.0, dict(FULL, n_train=4096, n_val=0, n_test=0),
                         splits=(0,))
        for mode in ("fp32", "fp16"):
            ac = torch.autocast("cuda", dtype=torch.float16) if mode == "fp16" else torch.autocast("cpu", enabled=False)
            with ac:
                das_steps(sets[0], FULL, torch.Generator().manual_seed(0), 3)      # warm-up
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                t0 = time.time()
                das_steps(sets[0], FULL, torch.Generator().manual_seed(1), 20)
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                step = (time.time() - t0) / 20
            # the run per model: 2 variables + the diagnostic, 3 restarts x 500 steps each
            per_model_h = 3 * 3 * 500 * step / 3600
            out[f"{name}|{mode}"] = {"seconds_per_step": step, "das_hours_per_model": per_model_h,
                                     "das_hours_9_seeds": 9 * per_model_h}
            print(f"  {name} {mode}: {step:.3f} s per step; DAS for 9 seeds about {9 * per_model_h:.1f} h", flush=True)
        if torch.cuda.is_available():
            out[f"{name}|peak_gpu_gb"] = torch.cuda.max_memory_allocated() / 1e9
        del hf, model
    out["environment"] = environment({"mode": "fp32 and fp16 autocast"})
    with open("pythia_address_time.json", "w") as f:
        json.dump(out, f, indent=1)
    return out


def units(records: list[dict], variable: str, arm: str, key: str = None) -> list[float]:
    out = []
    for seed in CHECK_SEEDS:
        vals = []
        for r in records:
            if r["seed"] != seed or r["size"] not in SIZES:
                continue
            rec = r[key or variable]
            if rec["n_informative"] >= MIN_INFORMATIVE:
                vals.append(rec[arm])
        out.append(float(np.nanmean(vals)) if vals else float("nan"))
    return out


def verdict(row: dict) -> str:
    lo, hi = row["ci95_difference"]
    if lo >= -MARGIN and hi <= MARGIN:
        return "ANALYTIC"
    if row["reject"] and row["delta"] > 0:
        return "DAS HIGHER"
    if row["reject"] and row["delta"] < 0:
        return "FORMULA HIGHER"
    return "UNDETERMINED"


def aggregate(records: list[dict]) -> dict:
    arms = ("das", "random") + FORMULAS
    u = {f"SUBJ@{pt}": {arm: units(records, "SUBJ", arm, key=key) for arm in arms} for pt, key in POINTS.items()}
    u["ATTR@last"] = {arm: units(records, "ATTR", arm) for arm in arms}
    # (a) existence: three comparisons, Holm over the three
    rows_a, ps = [], []
    for name in ("SUBJ@last", "SUBJ@arrival", "ATTR@last"):
        delta, p = paired_sign_flip(u[name]["das"], u[name]["random"])
        rows_a.append({"variable_point": name, "reference": "random", "delta": delta, "p": p})
        ps.append(p)
    for r, h in zip(rows_a, holm(ps)):
        r.update(p_adj=h["p_adj"], reject=bool(h["reject"] and r["delta"] > 0))
    read_a = {r["variable_point"]: ("CONFIRMED" if r["reject"] and np.nanmean(u[r["variable_point"]]["das"]) >= BAR
                                    else "NOT CONFIRMED") for r in rows_a}
    # (b) analytic: per point, DAS vs the six formulas, Holm over the six of that point
    rows_b, read_b = {}, {}
    for pt in POINTS:
        rows, ps = [], []
        for f in FORMULAS:
            a, b = u[f"SUBJ@{pt}"]["das"], u[f"SUBJ@{pt}"][f]
            delta, p = paired_sign_flip(a, b)
            diff = [x - y for x, y in zip(a, b) if np.isfinite(x) and np.isfinite(y)]
            rows.append({"formula": f, "uses_labels": f in LABELLED, "delta": delta, "p": p,
                         "ci95_difference": boot(diff)})
            ps.append(p)
        for r, h in zip(rows, holm(ps)):
            r.update(p_adj=h["p_adj"], reject=bool(h["reject"]))
            r["verdict"] = verdict(r)
        vs = [r["verdict"] for r in rows]
        read_b[pt] = ("ADDRESS ANALYTIC" if any(v in ("ANALYTIC", "FORMULA HIGHER") for v in vs)
                      else "DAS HIGHER THAN EVERY FORMULA" if all(v == "DAS HIGHER" for v in vs) else "MIXED")
        rows_b[pt] = rows
    r1_points = [pt for pt in POINTS if read_a[f"SUBJ@{pt}"] == "CONFIRMED"]
    means = {k: {arm: {"mean": float(np.nanmean(x)), "ci95": boot(x)} for arm, x in v.items()} for k, v in u.items()}
    per_size = {}
    for s in sorted({r["size"] for r in records}):
        per_size[s] = {}
        for name, key, var in (("SUBJ@last", "SUBJ", "SUBJ"), ("SUBJ@arrival", "SUBJ_arrival", "SUBJ"),
                               ("ATTR@last", "ATTR", "ATTR")):
            sel = [r[key] for r in records if r["size"] == s and r[key]["n_informative"] >= MIN_INFORMATIVE]
            per_size[s][name] = {arm: float(np.nanmean([x[arm] for x in sel])) if sel else float("nan") for arm in arms}
    diag = {arm: units(records, "SUBJ", arm, key="SUBJ_subject_position") for arm in ("das",) + FORMULAS}
    return {"unit_values": u, "means": means, "existence": rows_a, "read_out_existence": read_a,
            "analytic": rows_b, "read_out_analytic": read_b, "R1_points": r1_points,
            "stop": not r1_points, "per_size": per_size, "subject_position_units": diag,
            "subject_position_means": {arm: float(np.nanmean(x)) for arm, x in diag.items()}}


def pilot_summary(doc: dict) -> dict:
    """From the pilot file: per size the DAS profile at the last token (mean over that size's pilot
    models), the last layer r* (rule written before the pilot), the arrival layer (first r whose mean
    DAS >= ARRIVAL_BAR, criterion chosen after the pilot), the formulas at both points and the
    strongest formula without and with labels at each point (mean over the size, then over
    14m-160m)."""
    out = {"sizes": {}}
    for size in SIZES + (SIZE_410,):
        names = [n for n, m in doc["pilot"].items() if m["size"] == size]
        if not names:
            continue
        rs = sorted(int(r) for r in doc["pilot"][names[0]]["profile"])
        prof = {r: float(np.nanmean([doc["pilot"][n]["profile"][str(r)]["das"] for n in names])) for r in rs}
        last = max(rs, key=lambda r: (prof[r], -r))
        arrival = min(r for r in rs if prof[r] >= ARRIVAL_BAR)
        at = {pt: {k: float(np.nanmean([doc["pilot"][n]["profile"][str(r)][k] for n in names]))
                   for k in ("das",) + FORMULAS} for pt, r in (("last", last), ("arrival", arrival))}
        out["sizes"][size] = {"models": names, "profile": prof, "last": last, "arrival": arrival, "at": at}
    out["pooled"] = {pt: {k: float(np.nanmean([out["sizes"][s]["at"][pt][k] for s in SIZES]))
                          for k in ("das",) + FORMULAS} for pt in POINTS}
    out["strongest"] = {pt: {"unlabelled": max(UNLABELLED, key=lambda k: out["pooled"][pt][k]),
                             "labelled": max(LABELLED, key=lambda k: out["pooled"][pt][k])} for pt in POINTS}
    return out


def main(argv=None) -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--time", action="store_true")
    ap.add_argument("--pilot", action="store_true")
    ap.add_argument("--out", default="R0")
    ap.add_argument("--verify", default=None)
    ap.add_argument("--pilot-summary", default=None, help="print the choices derived from a pilot file")
    a = ap.parse_args(argv)
    if a.verify:
        return verify(a.verify)
    if a.pilot_summary:
        ps = pilot_summary(json.load(open(a.pilot_summary)))
        for size, v in ps["sizes"].items():
            print(f"{size:5s} last r={v['last']:2d} arrival r={v['arrival']:2d}  DAS by r: " +
                  " ".join(f"{r}:{x:.2f}" for r, x in v["profile"].items()))
        print("pooled 14m-160m:", {pt: {k: round(x, 3) for k, x in d.items()} for pt, d in ps["pooled"].items()})
        print("strongest:", ps["strongest"])
        same = ({s: v["last"] for s, v in ps["sizes"].items()} == R_STAR and
                {s: v["arrival"] for s, v in ps["sizes"].items()} == R_ARRIVAL and ps["strongest"] == STRONGEST)
        print("the protocol's constants agree with this file:", same)
        return 0 if same else 5
    set_determinism()
    torch.set_num_threads(4)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    started = datetime.now(timezone.utc).isoformat(timespec="seconds")
    t0 = time.time()
    if a.smoke:
        pilot(device, SMOKE, "", models=("pythia-14m",), smoke=True, parts="smoke_tmp/pilot_parts")
        run_model("pythia-14m-seed1", "14m", 1, 1, SMOKE, device, "smoke_tmp")
        print(f"smoke done ({time.time() - t0:.0f}s), nothing kept but smoke_tmp/")
        return 0
    if a.time:
        time_410(device)
        return 0
    if a.pilot:
        pilot(device, PILOT, "pythia_address_pilot.json")
        print(f"pilot done ({time.time() - t0:.0f}s)")
        return 0
    if not R_STAR or not R_ARRIVAL or WITH_410M is None or STRONGEST["last"]["unlabelled"] is None:
        print("the pilot's choices are not in the protocol yet. Stopping.")
        return 3
    os.makedirs(a.out, exist_ok=True)

    def provenance(records):
        envs = {json.dumps({k: v for k, v in r["environment"].items() if k != "precision"}, sort_keys=True)
                for r in records}
        return {"started_utc": started, "environments": [json.loads(e) for e in envs],
                "script_sha256": sha256_file(__file__), "protocol_commit": PROTOCOL_COMMIT,
                "protocol_script_sha256": PROTOCOL_SHA256, "revisions": REVISIONS,
                "r_last": R_STAR, "r_arrival": R_ARRIVAL, "r_diag": R_DIAG,
                "strongest_for_R1": STRONGEST, "with_410m": WITH_410M, "config": FULL,
                "streams": {"pairs": PAIR_STREAM, "formula": FORMULA_STREAM, "probe": PROBE_STREAM,
                            "random": RANDOM_STREAM, "torch": TORCH_STREAM},
                "unit": "seed (1..9), mean over 14m-160m"}
    # the confirmatory part: written once, as soon as the 36 models are done
    if os.path.exists(CONFIRMATORY_FILE):
        print(f"{CONFIRMATORY_FILE} exists; the confirmatory part is kept as it is")
    else:
        records = []
        for seed in CHECK_SEEDS:
            for size in SIZES:
                records.append(run_model(check_name(size, seed), size, seed, R_STAR[size], FULL, device, a.out))
        doc = {"provenance": provenance(records), "result": aggregate(records), "per_model": records,
               "wall_seconds": time.time() - t0}
        with open(CONFIRMATORY_FILE, "x") as f:
            json.dump(doc, f, indent=1)
        r = doc["result"]
        print("existence:", r["read_out_existence"], "| analytic:", r["read_out_analytic"],
              "| R.1 points:", r["R1_points"])
        print(f"wrote {CONFIRMATORY_FILE} ({time.time() - t0:.0f}s)", flush=True)
    # the descriptive part: 410m in fp16, its own file
    if WITH_410M and not os.path.exists(DESCRIPTIVE_410M_FILE):
        extra = [run_model(check_name(SIZE_410, seed), SIZE_410, seed, R_STAR[SIZE_410], FULL, device, a.out,
                           precision="fp16") for seed in CHECK_SEEDS]
        doc = {"provenance": provenance(extra), "descriptive_410m": extra, "wall_seconds": time.time() - t0}
        with open(DESCRIPTIVE_410M_FILE, "x") as f:
            json.dump(doc, f, indent=1)
        print(f"wrote {DESCRIPTIVE_410M_FILE} ({time.time() - t0:.0f}s)")
    return 0


def verify(path: str) -> int:
    """Locally, before a result is committed: every confirmatory record was computed by the script of the
    protocol commit; the file itself and every 410m record by this file as committed (Amendment 1)."""
    import subprocess
    here = os.path.relpath(__file__)
    show = lambda rev: subprocess.run(["git", "show", f"{rev}:{here}"], capture_output=True, check=True).stdout
    protocol = hashlib.sha256(show(PROTOCOL_COMMIT)).hexdigest()
    current = hashlib.sha256(show("HEAD")).hexdigest()
    doc = json.load(open(path))
    ok = protocol == PROTOCOL_SHA256 and doc["provenance"]["script_sha256"] == current
    for r in doc.get("per_model", []):
        ok &= r["environment"]["script_sha256"] in (protocol, current)
    for r in doc.get("descriptive_410m", []):
        ok &= r["environment"]["script_sha256"] == current
    recorded = sorted({r["environment"]["script_sha256"][:12]
                       for r in doc.get("per_model", []) + doc.get("descriptive_410m", [])})
    print(f"protocol {protocol[:12]}, current {current[:12]}; models {recorded}; file "
          f"{doc['provenance']['script_sha256'][:12]} -> {'OK' if ok else 'MISMATCH'}")
    return 0 if ok else 4


if __name__ == "__main__":
    sys.exit(main())
'''


In [ ]:
#@title 2.12 Протокол этапа R.1 (docstring) и импорты
PARTS[('pythia_interpreter.py', 0)] = r'''#!/usr/bin/env python
"""Stage R.1: the interpreter finds the address of the subject's number in unfamiliar Pythia models.

PROTOCOL. Committed to the repository before the main run, once (after the pilot below).
---------------------------------------------------------------------------
Role.       Stage 2.1 repeated on models trained by other people. R.0 (results/pythia_address.json,
            commit b420268) found the address: DAS's interchange R^2 is 0.839 at the last layer and
            0.559 at the arrival layer, and no formula gives it (at the last layer the strongest
            without labels reaches 0.420, the strongest with labels 0.535). R.1 tests the
            interpreter: can one trained model, from forward- and backward-pass signals, find that
            address in a model it has never seen, with no counterfactual pair from that model.
            DAS (Geiger, Wu, Potts, Icard and Goodman, CLeaR 2024) is only the upper bound (R.0's
            value) and the data curve, never a rival: it is trained on the interventions that define
            the correct answer. Training a network on DAS's own objective to output the subspace
            follows HyperDAS (Sun et al., ICLR 2025), which does it inside one model; here the network
            reads only forward- and backward-pass signals and is applied to models it never saw.
Variable.   Given by the task, as the input groups of part 1: the interpreter is told which variable to
            look for through the variable's carrier words (D below). Naming a cause in words
            (hypothesis component 5) is not tested here.
Models.     The 36 check models of R.0: PolyPythias (van der Wal et al., ICLR 2025)
            EleutherAI/pythia-{14m,31m,70m,160m}-seed{1..9}, revisions pinned in pythia_address.py.
            Cross-fitting by seed in three groups, {1,2,3}, {4,5,6}, {7,8,9}: the interpreter of group k
            is trained on the 24 models of the other two groups (all four sizes) and evaluated on the 12
            models of group k. So no model is read by an interpreter that saw any model of its seed;
            the models of one seed share the batch order across sizes, hence the split by seed. Every
            seed becomes a check unit (a 5/4 split would leave a smallest p of 0.0625).
            Pilot models only, never checked: seed 0 of each size and pythia-160m-{data,weight}-
            seed{1,2,3}, as in R.0.
            Descriptive, transfer to an unseen size: pythia-410m-seed{1..9} in fp32 (R.0's fp16 run
            overflowed in 7 of 9 seeds), read by the ensemble of all nine interpreters, against the
            formulas, without DAS (author's decision, 2026-10-19).
Points.     R.0's two main points, unchanged: the residual stream at the last position after the last
            layer (r = 6, 6, 6, 12 for 14m, 31m, 70m, 160m) and at the arrival layer (r = 6, 5, 5, 7),
            k = 1. In 14m the two coincide, and its arrival value is its last-layer value, as in R.0.
            R.0's caveat stands: the arrival layer of 160m was chosen on the pilot models, and on the
            check models DAS reaches only 0.268 there (no seed reaches 0.5). The definition is kept
            (author's decision, 2026-10-19), and every size is reported.
Signals.    Only quantities of forward and backward passes of the model on its own sentences: the
            4096 sentences on which R.0 built its formulas (pythia_address.FORMULA_STREAM). Per sentence,
            at the point: h, the residual; g = dB/dh; D = J . de, the derivative of the residual along
            de = e(plural form) - e(singular form) at the variable's carriers (de does not depend on the
            sentence's number, so it carries no label); and B(x). Never: the variable's labels, the
            model's counterfactual pairs, interventions, DAS, the words.
            Per coordinate j, each column v encoded as sign(v_j) log(1 + |v_j| / rms(v)), rms over j (a
            robust encoding of heavy-tailed statistics, the same for every width; chosen on the pilot
            models, see 'Pilot'): the mean and the spread of h_j, g_j and D_j; cov(h_j, g_j); cov(h_j, B);
            the coordinates of the six formulas without labels below, each sign-aligned with the point's
            start formula; the start formula.
            Global: log d / log 1024, r / L, mean B / spread of B, the share of sentences with B > 0,
            the absolute cosines between the six formulas without labels (15), and the shares of the
            five largest eigenvalues of cov(h).
            The coordinates are a set, with no positional encoding. The residual basis is not
            privileged in principle; a linear head over these features already gives any combination
            of the formulas' directions, so a rotation-free reader is a special case of this one.
Interpreter. As in 2.1: a transformer over the set of coordinates (2 layers, 4 heads, width 64,
            dropout 0.1) with a global context. Per coordinate w_j = softplus(gamma_j) s_j + psi_j and
            q = w / |w|, where s is the point's start formula divided by its largest magnitude. At
            initialisation gamma = softplus^-1(1) and psi = 0, so the untrained interpreter IS the start
            formula (a test checks it, and the pilot's epoch 0 reproduces the formula's R^2). One
            interpreter serves both points: a point is one more training item, r / L is in the global
            vector.
Training.   For each training (model, point): the first 1024 of R.0's 4096 training pairs and DAS's own
            objective, mean (((B(h') - B(x_cf)) / sigma_B)^2), h' = h_b + ((h_s - h_b) . q) q; 256 pairs
            per step, one (model, point) per step, the items in a random order each epoch. The loss
            depends on q only through q q^T (a test checks q -> -q). AdamW, lr 1e-3, weight decay 1e-2.
            Interpreter seeds 0, 1, 2; the ensemble address is the leading eigenvector of the mean of the
            three q q^T. The check group's pairs, DAS and labels are never used.
Start and epochs. Chosen on the pilot models only, by a rule written before the pilot:
            (1) the start of each point is the formula without labels with the highest mean R^2 over the
                10 pilot models on R.0's validation split (1024 pairs); no training;
            (2) the epochs, from 25, 50, ..., 300: halves P1 = {14m, 70m, 160m-data-seed1,
                160m-data-seed2, 160m-weight-seed1} and P2 = {31m, 160m, 160m-data-seed3,
                160m-weight-seed2, 160m-weight-seed3} (seed 0 unless named); interpreter seed 0 trained
                on one half and validated on the other, both ways; the epoch with the highest mean
                validation R^2 over the two halves and the two points (ties: the smaller).
            Recorded under 'Pilot' below before the protocol commit.
Arms.       On the check models, R.0's 2048 test pairs, interchange R^2; the number of the checked
            model's counterfactual pairs each arm uses is given on the right:
              interpreter                                                       0
              WITHOUT labels, from the interpreter's own signals:
                pca, first_order, gradient (R.0)                                0
                outcov    cov(h, B)                                             0
                outreg    (Sigma_h + lambda I)^-1 cov(h, B), lambda as R.0's LDA 0
                outprobe  R.0's probe with the label 1[B(x) > 0], the model's own choice
                                                                                0
              WITH LABELS of the variable on the checked model, which the interpreter does not get:
                diffmean, lda, probe (R.0)                                      0
              random: the mean over R.0's 10 random directions                  0
              DAS(n), n = 16, 64, 256, 1024 (the data curve, descriptive)       n
              DAS of R.0 (4096 training + 1024 validation), the upper bound     5120
            The three new formulas are R.0's diffmean, LDA and probe with the label replaced by the
            model's own output, which the interpreter receives. By the general rule of CLAUDE.md the
            strongest non-learned method of the level goes in before the rule; the model almost always
            agrees the verb with its subject, so outprobe may come close to the labelled probe.
Main family. Per point (last, arrival):
              (a) the interpreter vs the strongest formula without labels;
              (b) the interpreter vs the strongest formula with labels, non-inferiority margin 0.05.
            The strongest = the highest mean over the 9 check units in this run, among the six without
            labels and among the three with labels. Choosing after the run can only make the test
            harder. This replaces the rivals R.0 chose on its pilot (last layer gradient / probe, arrival
            first_order / diffmean): on the check models gradient (0.088 against -0.219) and probe
            (0.268 against -0.226) are the stronger ones at the arrival layer (author's decision,
            2026-10-19).
            (a): exact two-sided sign-flip test over the 9 units, positive difference required.
            (b): the lower one-sided test of TOST, an exact one-sided sign-flip test of
                 (interpreter - formula + 0.05) > 0 (author's decision, 2026-10-19: the upper test of a
                 full TOST would count an interpreter better than the formula by more than 0.05 as a
                 failure).
            Holm over these 4. With 9 units the smallest p is 2/512 = 0.0039 two-sided and 1/512 =
            0.0020 one-sided, so after Holm every comparison can still reject (at most 0.016).
Unit.       The seed (1..9); its value is the mean over 14m-160m of the model's test R^2, over the models
            with at least 50 informative test pairs (R.0's rule).
Read-out.   Per point: CONFIRMED if (a) rejects with a positive difference AND (b) rejects; NOT
            CONFIRMED otherwise, naming the comparison that did not reject.
            Admissible negative outcomes, written in advance: "the address exists (R.0), but the
            interpreter does not extract it from the signals better than the strongest formula without
            labels"; "it extracts it better than the formulas without labels, but a formula with the
            variable's labels is better by more than 0.05". Whether R^2 reaches 0.50 is descriptive.
            What the result shows, by hypothesis component: 1 one trained model (three seeds, an
            ensemble); 2 universal signals; 3 the address is B's own, judged by B's own counterfactual
            output; 4 accuracy against random, the formulas without labels and the marked formulas with
            labels; 5 NOT tested (the variable is given by the task); 6 transfer to unseen models (other
            seeds) under the main rule, to an unseen size (410m) and to another variable (ATTR)
            descriptively; to other model families (GPT-2, Qwen) in a later stage.
Descriptive, outside the family:
            the DAS data curve on the check models (the first n = 16, 64, 256, 1024 training pairs, R.0's
            recipe: lr 1e-2, batch 256, 500 steps, 3 restarts, the restart with the lowest training
            loss), giving "the interpreter is worth about n pairs";
            transfer to another variable: ATTR at the last layer, queried through ATTR's carriers by the
            interpreter trained on SUBJ only (R.0: DAS 0.634, the best formula 0.103);
            transfer to an unseen size: 410m at both points (r = 24 and 14) against the formulas (for
            reference, R.0's two 410m seeds without fp16 overflow gave DAS 0.93 at the last layer);
            every size, every template, the decision IIA, each interpreter seed.
Statistics. 95% percentile bootstrap intervals over the units (10 000 resamples, numpy default_rng(0)).
Controls.   A failure stops the run.
            (1) Leak: the interpreter's inputs are built by a function that receives only the model and
                its own sentences; a test builds them with the pair generator, the pair sets and DAS
                disabled and with the labels shuffled, and finds them unchanged. The model's outputs on
                ordinary sentences are not interventions; the formulas receive them too.
            (2) Reproduction of R.0 on every check model: the test R^2 of R.0's six formulas and of the
                random directions at SUBJ last, SUBJ arrival and ATTR last equal R.0's values
                (|difference| <= 1e-3 (1 + |value|)), and sigma_B to 1e-4 relative; this also confirms
                the pairs and the revisions. R.0's float64 controls (1)-(3) at both points, every model.
            (3) The untrained interpreter equals its start formula (a test; and the pilot's epoch 0
                reproduces the start formula's validation R^2 to 1e-5 (1 + |R^2|), fp32 rounding).
            (4) The loss is unchanged by q -> -q (a test).
Precision.  fp32, TF32 off, R.0's determinism flags, recorded.
Provenance. Every file records this script's sha256 and pythia_address.py's, the sha256 of R.0's result
            file, the versions, the GPU and the flags; the interpreters' weights are saved with their
            hashes. `--verify` checks locally that the run used the script of the protocol commit before
            the result is committed.
Pilot.      Before the pilot, 2026-10-19: a local diagnostic of the code on three pilot models only
            (pythia-14m, -31m, -70m, seed 0; CPU; the start by rule (1) on these three: outreg at the
            last layer, first_order at the arrival layer; interpreter seed 0 trained on two models and
            validated on the third, each way, 200 epochs, validation R^2 every 25). With every feature
            column divided by its largest magnitude, as in 2.1, held-out R^2 (last / arrival) went from
            the start to: 14m 0.442 -> 0.68-0.77 from epoch 25 on; 31m 0.579 -> 0.66-0.68 and
            0.265 -> 0.42-0.44; 70m arrival 0.437 -> 0.54-0.57, but 70m last 0.664 -> 0.691 at epoch 25
            and then -0.48 to -1.27. A rule written before the second run: adopt the encoding
            sign(v) log(1 + |v| / rms(v)) if its mean held-out R^2 over the five (model, point) values
            and the epochs 25-200 is higher. It was: 0.457 against 0.344 (14m 0.68-0.77; 31m 0.60-0.68
            and 0.43-0.45; 70m arrival 0.56-0.58; 70m last still -0.74 to +0.19). So the encoding is
            adopted. Trained only on smaller models, the interpreter fails at 70m's last layer under
            either encoding; the main design trains every group on all four sizes, while 410m
            (descriptive) is such an extrapolation. The check models were not touched.
            The pilot, 2026-10-19, on Colab (Tesla T4, torch 2.11.0+cu130, transformers 5.18.0) with this
            script at commit 5341399 (sha256 cb9199b2...), 845 s; results/pythia_interpreter_pilot.json.
            R.0's float64 controls passed on every pilot model (worst 4.0e-12). Control (3): the untrained
            interpreter reproduced its start formula's validation R^2 to 3.1e-6 (1 + |R^2|).
            (1) Mean validation R^2 over the 10 pilot models, last / arrival layer:
                without labels: pca -0.001 / -0.004, first_order 0.446 / 0.379, gradient 0.700 / 0.392,
                outcov -0.799 / 0.159, outreg 0.586 / 0.116, outprobe 0.717 / 0.303;
                with labels: diffmean 0.456 / 0.440, lda 0.084 / 0.046, probe 0.700 / 0.405.
                By the rule: START = outprobe at the last layer, gradient at the arrival layer.
            (2) Mean validation R^2 over the two halves and the two points, by epoch: 25 0.639, 50 0.673,
                75 0.675, 100 0.675, 125 0.671, 150 0.670, 175 0.665, 200 0.668, 225 0.667, 250 0.677,
                275 0.679, 300 0.668. By the rule: EPOCHS = 275 (flat from epoch 50 on).
                Per half, from epoch 0 (the start formula) to epoch 275, last and arrival: trained on P1,
                validated on P2, 0.760 -> 0.783 and 0.372 -> 0.555; trained on P2, validated on P1,
                0.674 -> 0.770 and 0.418 -> 0.606.
            Before this commit, control (2) alone was run locally on one check model (pythia-14m-seed1,
            CPU): R.0's formulas were reproduced to 1.5e-4 relative and sigma_B to 2.7e-6; no interpreter
            was run on any check model.
---------------------------------------------------------------------------
Colab (T4), from the notebook colab/R1_pythia.ipynb:
    !python pythia_interpreter.py --smoke                    # minutes, writes only smoke_tmp/
    !python pythia_interpreter.py --pilot                    # writes pythia_interpreter_pilot.json
    !python pythia_interpreter.py --pilot-summary pythia_interpreter_pilot.json
    !python pythia_interpreter.py --out R1                   # the main run: pythia_interpreter.json
    !python pythia_interpreter.py --curve --out R1           # descriptive: pythia_interpreter_curve.json
    !python pythia_interpreter.py --m410 --out R1            # descriptive: pythia_interpreter_410m.json
Locally, before committing a result:
    python scripts/pythia_interpreter.py --verify pythia_interpreter.json
"""
from __future__ import annotations

import argparse
import gc
import hashlib
import json
import math
import os
import subprocess
import sys
import time
import types
from datetime import datetime, timezone
from itertools import product

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import pythia_address as P  # noqa: E402  (stage R.0: models, pairs, forward pass, formulas, statistics)

'''


In [ ]:
#@title 2.13 Константы
PARTS[('pythia_interpreter.py', 1)] = r'''# ------------------------------------------------------------------ constants
FOLDS = ((1, 2, 3), (4, 5, 6), (7, 8, 9))
POINTS = ("last", "arrival")
UNLAB = ("pca", "first_order", "gradient", "outcov", "outreg", "outprobe")
LAB = ("diffmean", "lda", "probe")
R0_KEY = {"last": "SUBJ", "arrival": "SUBJ_arrival", "attr": "ATTR"}
DRAW = {0: P.FULL["n_train"], 1: P.FULL["n_val"], 2: P.FULL["n_test"]}     # R.0's draws of each split
FULL = dict(n_formula=P.N_FORMULA, n_train=1024, n_val=1024, n_test=2048, seeds=(0, 1, 2),
            curve_n=(16, 64, 256, 1024), curve_steps=500, curve_restarts=3)
SMOKE = dict(n_formula=64, n_train=64, n_val=32, n_test=32, seeds=(0,), epochs=2,
             curve_n=(16,), curve_steps=3, curve_restarts=1)
PAIR_BATCH, LR, WEIGHT_DECAY = 256, 1e-3, 1e-2
EPOCH_GRID = tuple(range(25, 301, 25))
PILOT_HALVES = (("pythia-14m", "pythia-70m", "pythia-160m-data-seed1", "pythia-160m-data-seed2",
                 "pythia-160m-weight-seed1"),
                ("pythia-31m", "pythia-160m", "pythia-160m-data-seed3", "pythia-160m-weight-seed2",
                 "pythia-160m-weight-seed3"))
PILOT_SEED = 0
MARGIN = 0.05
R0_TOL, SIGMA_TOL, START_TOL = 1e-3, 1e-4, 1e-5
INTERP_STREAM, ORDER_STREAM, CURVE_STREAM = 3040, 3041, 3042
SOFTPLUS_INV_1 = math.log(math.e - 1.0)
R0_FILE = "pythia_address.json"
R0_SHA256 = "8c802368371d22115b9bf38d3d828e6c3d9bcfb95294afd76de1719145351a6e"
OUT_MAIN, OUT_CURVE, OUT_410, OUT_PILOT = ("pythia_interpreter.json", "pythia_interpreter_curve.json",
                                         "pythia_interpreter_410m.json", "pythia_interpreter_pilot.json")
PROTOCOL_SUBJECT = "Protocol, stage R.1"
# fixed by the pilot before the protocol commit (see 'Pilot'); a test checks them against the pilot file
START = {"last": "outprobe", "arrival": "gradient"}
EPOCHS = 275


'''


In [ ]:
#@title 2.14 Сигналы и формулы (без пар)
PARTS[('pythia_interpreter.py', 2)] = r'''# ------------------------------------------------------------------ signals and formulas (no pairs)
def own_sentences(n: int) -> np.ndarray:
    """R.0's formula sentences: the model's own distribution, the same for every model."""
    return P.sample(n, np.random.default_rng([P.FORMULA_STREAM]))


def point_signals(model: P.Neox, vocab: dict, variable: str, r: int, n: int) -> tuple[dict, torch.Tensor]:
    """At (r, last position), on the model's own sentences: h, g = dB/dh, D = J . de at the variable's
    carriers and B(x), the interpreter's raw signals; returned apart, the variable's labels, which only
    the labelled formulas use. Receives only the model and its sentences: no pairs, no interventions."""
    recs = own_sentences(n)
    fi = P.formula_inputs(model, recs, vocab, variable, r, "last")
    order = torch.as_tensor(np.concatenate(list(P.by_template(recs).values())))
    B = P.outputs(model, recs, vocab).double().cpu()[order]          # the row order of formula_inputs
    return {"H": fi["H"], "G": fi["G"], "D": fi["D"], "B": B}, fi["y"]


def _unit(v: torch.Tensor) -> torch.Tensor:
    return v / v.norm().clamp(min=1e-30)


def probe_direction(H: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """R.0's probe line for line (pythia_address.formulas); a test checks the equality."""
    d = H.shape[1]
    Hc = H - H.mean(0)
    scale = Hc.pow(2).mean().sqrt().clamp(min=1e-30)
    X = Hc / scale
    w = torch.zeros(d, dtype=X.dtype, requires_grad=True)
    b = torch.zeros(1, dtype=X.dtype, requires_grad=True)
    opt = torch.optim.LBFGS([w, b], max_iter=500, tolerance_grad=1e-10, tolerance_change=1e-12,
                            line_search_fn="strong_wolfe")
    sgn = 2 * y - 1

    def closure():
        opt.zero_grad()
        loss = F.softplus(-sgn * (X @ w + b)).mean() + 0.5e-3 * (w * w).sum()
        loss.backward()
        return loss
    opt.step(closure)
    return _unit(w.detach())


def unlabelled_formulas(sig: dict) -> dict:
    """The six formulas without labels: R.0's pca, first_order and gradient, computed as R.0 computes
    them, and outcov, outreg and outprobe from the model's own output B(x)."""
    H, G, D, B = sig["H"], sig["G"], sig["D"], sig["B"]
    n, d = H.shape
    Hc = H - H.mean(0)
    cov = Hc.T @ Hc / n
    c = Hc.T @ (B - B.mean()) / n
    lam = 1e-3 * torch.trace(cov) / d
    return {"pca": _unit(torch.linalg.eigh(cov)[1][:, -1]),
            "first_order": _unit(D.mean(0)),
            "gradient": _unit(G.mean(0)),
            "outcov": _unit(c),
            "outreg": _unit(torch.linalg.solve(cov + lam * torch.eye(d, dtype=cov.dtype), c)),
            "outprobe": probe_direction(H, (B > 0).to(H.dtype))}


def labelled_formulas(sig: dict, y: torch.Tensor) -> dict:
    """R.0's three formulas WITH the variable's labels, exactly as R.0 computes them."""
    full = P.formulas({"H": sig["H"], "G": sig["G"], "D": sig["D"], "y": y})
    return {k: full[k] for k in LAB}


'''


In [ ]:
#@title 2.15 Входы интерпретатора
PARTS[('pythia_interpreter.py', 3)] = r'''# ------------------------------------------------------------------ the interpreter's inputs
COLS = ("h_mean", "h_sd", "g_mean", "g_sd", "d_mean", "d_sd", "cov_hg", "cov_hb")
N_FEATURES = len(COLS) + len(UNLAB) + 1
N_GLOBAL = 4 + len(UNLAB) * (len(UNLAB) - 1) // 2 + 5


def point_stats(sig: dict, r: int, L: int) -> dict:
    """Everything the interpreter reads about one (model, point, variable), from the raw signals only."""
    H, G, D, B = sig["H"], sig["G"], sig["D"], sig["B"]
    n, d = H.shape
    Hc, Gc, Bc = H - H.mean(0), G - G.mean(0), B - B.mean()
    cols = {"h_mean": H.mean(0), "h_sd": H.std(0), "g_mean": G.mean(0), "g_sd": G.std(0),
            "d_mean": D.mean(0), "d_sd": D.std(0), "cov_hg": (Hc * Gc).mean(0), "cov_hb": Hc.T @ Bc / n}
    ev = torch.linalg.eigvalsh(Hc.T @ Hc / n).flip(0).clamp(min=0)
    eig = (ev[:5] / ev.sum().clamp(min=1e-30)).tolist()
    glob = [math.log(d) / math.log(1024), r / L, float(B.mean() / B.std().clamp(min=1e-30)),
            float((B > 0).double().mean())]
    return {"cols": cols, "dirs": unlabelled_formulas(sig), "glob": glob, "eig": eig + [0.0] * (5 - len(eig)),
            "d": d}


def _scaled(v: torch.Tensor) -> torch.Tensor:
    """Linear: divided by the largest magnitude (the start formula, so that the untrained interpreter is it)."""
    m = float(v.abs().max())
    return v / m if m > 0 else torch.zeros_like(v)


def _encode(v: torch.Tensor) -> torch.Tensor:
    """A feature column: sign(v) log(1 + |v| / rms(v)), rms over the coordinates; robust to heavy tails and
    the same for every width d (chosen on the pilot models, see 'Pilot')."""
    rms = float(v.pow(2).mean().sqrt())
    return torch.sign(v) * torch.log1p(v.abs() / rms) if rms > 0 else torch.zeros_like(v)


def inputs(st: dict, start: str, device) -> dict:
    """The interpreter's input for one (model, point): per-coordinate features, the global vector and
    the start formula, all from point_stats."""
    s = st["dirs"][start]
    aligned = {k: (v if float(v @ s) >= 0 else -v) for k, v in st["dirs"].items()}
    X = torch.stack([_encode(st["cols"][k]) for k in COLS] + [_encode(aligned[k]) for k in UNLAB] +
                    [_encode(s)], 1)
    cos = [abs(float(st["dirs"][a] @ st["dirs"][b])) for i, a in enumerate(UNLAB) for b in UNLAB[i + 1:]]
    u = torch.tensor(st["glob"] + cos + st["eig"], dtype=torch.float32)
    return {"X": X.float().to(device), "u": u.to(device), "s_hat": _scaled(s).float().to(device),
            "s": s.float().to(device)}


'''


In [ ]:
#@title 2.16 Интерпретатор и его обучение
PARTS[('pythia_interpreter.py', 4)] = r'''# ------------------------------------------------------------------ the interpreter and its training
def _enc(d_in: int, width: int, layers: int, heads: int, dropout: float):
    layer = nn.TransformerEncoderLayer(width, heads, 4 * width, dropout, activation="gelu",
                                       batch_first=True, norm_first=True)
    return (nn.Sequential(nn.Linear(d_in, width), nn.GELU(), nn.Linear(width, width)),
            nn.TransformerEncoder(layer, layers, enable_nested_tensor=False))


class Reader(nn.Module):
    """A transformer over the set of coordinates of the residual stream at one point; returns the unit
    direction q. Starts exactly at the start formula: softplus(gamma) = 1, psi = 0."""

    def __init__(self, n_features: int = N_FEATURES, n_global: int = N_GLOBAL, width: int = 64,
                 layers: int = 2, heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.c_in, self.c_enc = _enc(n_features, width, layers, heads, dropout)
        self.g_in = nn.Sequential(nn.Linear(n_global, width), nn.GELU())
        self.mix = nn.Sequential(nn.Linear(2 * width, width), nn.GELU())
        self.gamma = nn.Sequential(nn.Linear(2 * width, width), nn.GELU(), nn.Linear(width, 1))
        self.psi = nn.Sequential(nn.Linear(2 * width, width), nn.GELU(), nn.Linear(width, 1))
        nn.init.zeros_(self.gamma[-1].weight)
        nn.init.constant_(self.gamma[-1].bias, SOFTPLUS_INV_1)
        nn.init.zeros_(self.psi[-1].weight)
        nn.init.zeros_(self.psi[-1].bias)

    def forward(self, x: dict) -> torch.Tensor:
        h = self.c_enc(self.c_in(x["X"])[None])[0]                          # (d, width)
        ctx = self.mix(torch.cat([h.mean(0), self.g_in(x["u"])]))
        z = torch.cat([h, ctx[None].expand_as(h)], 1)
        w = F.softplus(self.gamma(z)[:, 0]) * x["s_hat"] + self.psi(z)[:, 0]
        return w / w.norm().clamp(min=1e-30)


def das_loss(ps: P.PairSet, q: torch.Tensor, rows: np.ndarray) -> torch.Tensor:
    """DAS's own objective (pythia_address.das_steps) for the direction q on the given rows."""
    dev = ps.model.device
    loss = 0.0
    for t, ix in ps.groups.items():
        sel = rows[np.isin(rows, ix)]
        if len(sel) == 0:
            continue
        loc = torch.as_tensor(ps.local[sel], device=dev)
        hb = ps.cache[t]["h"][loc, 0]
        hs = ps.h_s[t][loc]
        new = hb + ((hs - hb) @ q)[:, None] * q[None]
        pred = ps.model.cont(ps.cache[t], new, idx=loc)
        ref = ps.y_cf[torch.as_tensor(sel, device=dev)]
        loss = loss + (((pred - ref) / ps.sigma) ** 2).sum()
    return loss / len(rows)


def train_reader(items: list[dict], seed_parts: tuple, epochs: int, device, every: int = 0, log=None) -> Reader:
    """One interpreter on the training items, one (model, point) per step; log(epoch, reader) is called
    at epoch 0 and every `every` epochs."""
    torch.manual_seed(P.seed_of(INTERP_STREAM, *seed_parts))
    m = Reader().to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    g = torch.Generator().manual_seed(P.seed_of(ORDER_STREAM, *seed_parts))
    if every and log is not None:
        m.eval()
        log(0, m)
    for ep in range(1, epochs + 1):
        m.train()
        for i in torch.randperm(len(items), generator=g).tolist():
            ps = items[i]["pairs"]
            rows = torch.randint(0, ps.n, (min(PAIR_BATCH, ps.n),), generator=g).numpy()
            loss = das_loss(ps, m(items[i]["x"]), rows)
            opt.zero_grad()
            loss.backward()
            opt.step()
        if every and log is not None and ep % every == 0:
            m.eval()
            log(ep, m)
    m.eval()
    return m


def ensemble(qs: list[torch.Tensor]) -> torch.Tensor:
    """The leading eigenvector of the mean of the projectors q q^T."""
    M = torch.stack([torch.outer(q, q) for q in qs]).mean(0).double().cpu()
    return torch.linalg.eigh(M)[1][:, -1].float().to(qs[0].device)


@torch.no_grad()
def read(readers: list[Reader], x: dict) -> tuple[torch.Tensor, list[torch.Tensor]]:
    qs = [m(x) for m in readers]
    return ensemble(qs), qs


'''


In [ ]:
#@title 2.17 Одна модель: контроли, пары, формулы
PARTS[('pythia_interpreter.py', 5)] = r'''# ------------------------------------------------------------------ one model
def pair_set(model: P.Neox, vocab: dict, variable: str, r: int, sigma: float, split: int, n_keep: int) -> P.PairSet:
    """R.0's pairs: the first n_keep of the split as R.0 drew it (4096 training, 1024 validation and
    2048 test pairs), so every model and every arm sees the same pairs."""
    base, src, cf = P.make_pairs(variable, split, DRAW[split])
    base, src, cf = base[:n_keep], src[:n_keep], cf[:n_keep]
    if not P.differs_only_at_carriers(base, cf, variable, vocab):
        raise SystemExit("R.0's control (4): a minimal pair differs outside the carriers. Stopping.")
    return P.PairSet(model, vocab, base, src, cf, r, "last", sigma)


def trim(model: P.Neox, r_min: int) -> None:
    """Keep what continuing from index r_min needs (blocks r_min..L-1, the final norm, the readout row);
    the embeddings, the earlier blocks and the output matrix are released with the transformers model."""
    model.blocks = [None] * r_min + list(model.blocks[r_min:])
    model.m = types.SimpleNamespace(final_layer_norm=model.m.final_layer_norm)
    model.hf = None


def prepare(name: str, size: str, seed: int, device, c: dict, splits: tuple, r0: dict | None = None,
            attr: bool = False, loaded: tuple | None = None, rs: dict | None = None) -> dict:
    """One model: R.0's float64 controls; per point (and ATTR at the last layer if `attr`) the
    interpreter's statistics, every formula, the pair sets of `splits` (0 training, 1 validation, 2 test)
    and every formula's R^2 on the evaluation split; on check models R.0's reproduction control. Then
    the model is trimmed to the tail its pair sets need. `loaded` and `rs` are for the tests."""
    t0 = time.time()
    hf, model, vocab, rev = loaded or P.load(name, device)
    rs = rs or {"last": P.R_STAR[size], "arrival": P.R_ARRIVAL[size]}
    loaded = None
    ctl = {f"r{r}": P.controls(hf, model, vocab, r) for r in sorted(set(rs.values()))}
    for k, v in ctl.items():
        if P.controls_failed(v):
            raise SystemExit(f"{name} {k}: R.0's control failed {v}. Stopping.")
    sigma = P.probe_sigma(model, vocab)
    keep = {0: c["n_train"], 1: c["n_val"], 2: c["n_test"]}
    pm = {"model": name, "size": size, "seed": seed, "revision": rev, "sigma": sigma, "controls": ctl,
          "r": rs, "points": {}}
    queries = [("last", "SUBJ"), ("arrival", "SUBJ")] + ([("attr", "ATTR")] if attr else [])
    for key, var in queries:
        r = rs["arrival"] if key == "arrival" else rs["last"]
        if key == "arrival" and r == rs["last"]:
            continue                                                    # 14m: the two points coincide
        sig, y = point_signals(model, vocab, var, r, c["n_formula"])
        st = point_stats(sig, r, model.L)
        dirs = dict(st["dirs"])
        dirs.update(labelled_formulas(sig, y))
        it = {"key": key, "variable": var, "r": r, "stats": st, "dirs": dirs, "sets": {}}
        for split in splits:
            if key != "attr" or split == 2:                             # ATTR: evaluation only
                it["sets"][split] = pair_set(model, vocab, var, r, sigma, split, keep[split])
        ev = it["sets"].get(2) or it["sets"].get(1)
        if ev is not None:
            it["formula_r2"] = {k: ev.r2(v.to(device, torch.float32))[0] for k, v in dirs.items()}
            if 2 in it["sets"]:
                it["formula_r2"]["random"] = float(np.nanmean(
                    [ev.r2(v.to(device, torch.float32))[0] for v in P.random_dirs(model.d, size, seed, var)]))
            it["n_informative"] = ev.informative()
        pm["points"][key] = it
    if r0 is not None:
        pm["r0_check"] = r0_check(pm, r0)
        if not pm["r0_check"]["passed"]:
            raise SystemExit(f"{name}: R.0 is not reproduced {pm['r0_check']}. Stopping.")
    trim(model, min(rs.values()))
    del hf
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        pm["gpu_gb_after"] = torch.cuda.memory_allocated() / 1e9
    pm["seconds_prepare"] = time.time() - t0
    return pm


def r0_check(pm: dict, r0rec: dict) -> dict:
    """Control (2): R.0's six formulas and the random directions at every query, and sigma_B."""
    worst = 0.0
    for key, it in pm["points"].items():
        ref = r0rec[R0_KEY[key]]
        for f in P.FORMULAS + ("random",):
            worst = max(worst, abs(it["formula_r2"][f] - ref[f]) / (1 + abs(ref[f])))
    dev = abs(pm["sigma"] - r0rec["sigma"]) / r0rec["sigma"]
    return {"max_relative_deviation": worst, "sigma_relative_deviation": dev,
            "passed": bool(worst <= R0_TOL and dev <= SIGMA_TOL)}


def items_of(pm: dict, start: dict, device, split: int) -> list[dict]:
    """The training (or validation) items of a prepared model: one per point it has."""
    return [{"model": pm["model"], "size": pm["size"], "seed": pm["seed"], "point": key,
             "x": inputs(pm["points"][key]["stats"], start[key], device), "pairs": pm["points"][key]["sets"][split]}
            for key in POINTS if key in pm["points"]]


def evaluate(pm: dict, readers: list[Reader], start: dict, device) -> dict:
    """The ensemble address of a prepared model at every query, on its test pairs, with every formula."""
    rec = {k: pm[k] for k in ("model", "size", "seed", "revision", "sigma", "controls", "r")}
    rec["r0_check"] = pm.get("r0_check")
    rec["seconds_prepare"] = pm.get("seconds_prepare")
    for key, it in pm["points"].items():
        st_key = "last" if key == "attr" else key
        x = inputs(it["stats"], start[st_key], device)
        q, qs = read(readers, x)
        ev = it["sets"][2]
        r2, n_inf, dec = ev.r2(q)
        row = {"r": it["r"], "variable": it["variable"], "start": start[st_key], "interpreter": r2,
               "n_informative": n_inf, "decision_iia": dec, "by_template": ev.r2_by_template(q),
               "interpreter_seeds": [ev.r2(qi)[0] for qi in qs], "abs_cos_to_start": abs(float(q @ x["s"]))}
        row.update(it["formula_r2"])
        rec[key] = row
    if "arrival" not in pm["points"]:
        rec["arrival"] = dict(rec["last"], same_as_last=True)          # 14m, as in R.0
    return rec


'''


In [ ]:
#@title 2.18 Статистика и правило вывода
PARTS[('pythia_interpreter.py', 6)] = r'''# ------------------------------------------------------------------ statistics and the read-out
def one_sided_sign_flip(d) -> tuple[float, float]:
    """Exact one-sided sign-flip test of mean(d) > 0: (mean, p)."""
    d = np.asarray(d, dtype=float)
    d = d[np.isfinite(d)]
    if len(d) == 0:
        return float("nan"), float("nan")
    signs = np.array(list(product([1, -1], repeat=len(d))), dtype=float)
    null = (signs * d).mean(axis=1)
    return float(d.mean()), float((null >= d.mean() - 1e-12).mean())


def unit_values(records: list[dict], key: str, arm: str) -> list[float]:
    """Per seed 1..9: the mean over 14m-160m of the arm's test R^2 (models with >= 50 informative pairs)."""
    out = []
    for seed in P.CHECK_SEEDS:
        vals = [r[key][arm] for r in records if r["seed"] == seed and r["size"] in P.SIZES
                and r[key]["n_informative"] >= P.MIN_INFORMATIVE]
        out.append(float(np.nanmean(vals)) if vals else float("nan"))
    return out


def strongest(u: dict, arms: tuple) -> str:
    """The arm with the highest mean over the units (ties: the first in the protocol's order)."""
    return max(arms, key=lambda a: (float(np.nanmean(u[a])), -arms.index(a)))


def aggregate(records: list[dict]) -> dict:
    arms = ("interpreter",) + UNLAB + LAB + ("random", "das")
    u = {key: {arm: unit_values(records, key, arm) for arm in arms} for key in POINTS + ("attr",)}
    rows, ps = [], []
    rivals = {}
    for pt in POINTS:
        su, sl = strongest(u[pt], UNLAB), strongest(u[pt], LAB)
        rivals[pt] = {"without_labels": su, "with_labels": sl}
        da, pa = P.paired_sign_flip(u[pt]["interpreter"], u[pt][su])
        diff_a = [x - y for x, y in zip(u[pt]["interpreter"], u[pt][su]) if np.isfinite(x) and np.isfinite(y)]
        diff_b = [x - y for x, y in zip(u[pt]["interpreter"], u[pt][sl]) if np.isfinite(x) and np.isfinite(y)]
        shifted, pb = one_sided_sign_flip([x + MARGIN for x in diff_b])
        rows.append({"point": pt, "condition": "a", "rival": su, "uses_labels": False, "delta": da,
                     "p": pa, "ci95_difference": P.boot(diff_a)})
        rows.append({"point": pt, "condition": "b", "rival": sl, "uses_labels": True,
                     "delta": float(np.mean(diff_b)) if diff_b else float("nan"), "margin": MARGIN,
                     "shifted_delta": shifted, "p": pb, "ci95_difference": P.boot(diff_b)})
        ps += [pa, pb]
    for row, h in zip(rows, P.holm(ps)):
        positive = row["delta"] > 0 if row["condition"] == "a" else row["shifted_delta"] > 0
        row.update(p_adj=h["p_adj"], reject=bool(h["reject"] and positive))
    read_out = {}
    for pt in POINTS:
        failed = [f"({r['condition']}) vs {r['rival']}" for r in rows if r["point"] == pt and not r["reject"]]
        read_out[pt] = "CONFIRMED" if not failed else "NOT CONFIRMED: " + "; ".join(failed)
    means = {key: {arm: {"mean": float(np.nanmean(x)), "ci95": P.boot(x)} for arm, x in v.items()}
             for key, v in u.items()}
    per_size = {s: {key: {arm: float(np.nanmean([r[key][arm] for r in records if r["size"] == s
                                                 and r[key]["n_informative"] >= P.MIN_INFORMATIVE] or [np.nan]))
                          for arm in arms} for key in POINTS + ("attr",)} for s in P.SIZES}
    per_template = {key: {t: float(np.nanmean([r[key]["by_template"].get(t, np.nan) for r in records]))
                          for t in P.TEMPLATES} for key in POINTS + ("attr",)}
    seeds_mean = {key: float(np.nanmean([np.mean(r[key]["interpreter_seeds"]) for r in records]))
                  for key in POINTS + ("attr",)}
    return {"unit_values": u, "means": means, "rivals": rivals, "main_family": rows, "read_out": read_out,
            "per_size": per_size, "interpreter_by_template": per_template,
            "interpreter_single_seed_mean": seeds_mean,
            "decision_iia": {key: float(np.nanmean([r[key]["decision_iia"] for r in records]))
                             for key in POINTS + ("attr",)},
            "reaches_0.50": {pt: bool(np.nanmean(u[pt]["interpreter"]) >= 0.50) for pt in POINTS}}


'''


In [ ]:
#@title 2.19 Пилот
PARTS[('pythia_interpreter.py', 7)] = r'''# ------------------------------------------------------------------ the pilot
def validation_mean(prepared: dict, names: tuple, pt: str, r2_of) -> float:
    """Mean over the named models of r2_of(model, key) at the point pt, over the models with at least 50
    informative validation pairs; 14m's arrival is its last layer."""
    vals = []
    for n in names:
        key = pt if pt in prepared[n]["points"] else "last"
        if prepared[n]["points"][key]["n_informative"] >= P.MIN_INFORMATIVE:
            vals.append(r2_of(n, key))
    return float(np.nanmean(vals)) if vals else float("nan")


def start_rule(val: dict) -> dict:
    """(1) per point, the formula without labels with the highest mean validation R^2."""
    return {pt: max(UNLAB, key=lambda f: (val[pt][f], -UNLAB.index(f))) for pt in POINTS}


def epoch_rule(logs: list[dict]) -> tuple[int, dict]:
    """(2) the logged epoch with the highest mean validation R^2 over both halves and both points
    (ties: the smaller)."""
    grid = [e for e in EPOCH_GRID if all(e in lg for lg in logs)] or [max(lg for lg in logs[0])]
    mean = {e: float(np.mean([lg[e][pt] for lg in logs for pt in POINTS])) for e in grid}
    return max(grid, key=lambda e: (mean[e], -e)), mean


def pilot(device, c: dict, out: str, halves: tuple = PILOT_HALVES, epochs_max: int = max(EPOCH_GRID),
          every: int = 25) -> dict:
    """The rule written before the pilot: the start per point, then the epochs; pilot models only."""
    t0 = time.time()
    prepared = {}
    for name in halves[0] + halves[1]:
        prepared[name] = prepare(name, name.split("-")[1], 0, device, c, splits=(0, 1))
        print(f"  pilot {name}: prepared ({prepared[name]['seconds_prepare']:.0f}s)", flush=True)
    names = tuple(prepared)
    val = {pt: {f: validation_mean(prepared, names, pt, lambda n, key, f=f: prepared[n]["points"][key]["formula_r2"][f])
                for f in UNLAB + LAB} for pt in POINTS}
    start = start_rule(val)
    print("  start per point:", start, flush=True)
    logs, start_dev = [], 0.0
    for h in (0, 1):
        train_items = [it for n in halves[h] for it in items_of(prepared[n], start, device, 0)]
        val_names = halves[1 - h]
        xs = {(n, key): inputs(prepared[n]["points"][key]["stats"], start[key], device)
              for n in val_names for key in prepared[n]["points"]}
        log = {}

        def cb(ep, m, h=h, log=log, val_names=val_names, xs=xs):
            nonlocal start_dev
            with torch.no_grad():
                r2 = {nk: prepared[nk[0]]["points"][nk[1]]["sets"][1].r2(m(x))[0] for nk, x in xs.items()}
            if ep == 0:                                                   # control (3)
                for (n, key), v in r2.items():
                    ref = prepared[n]["points"][key]["formula_r2"][start[key]]
                    start_dev = max(start_dev, abs(v - ref) / (1 + abs(ref)))
            log[ep] = {pt: validation_mean(prepared, val_names, pt, lambda n, key: r2[(n, key)]) for pt in POINTS}
            print(f"  pilot half {h + 1} epoch {ep}: " + " ".join(f"{pt} {v:+.3f}" for pt, v in log[ep].items()),
                  flush=True)
        train_reader(train_items, (9, h, PILOT_SEED), epochs_max, device, every=every, log=cb)
        logs.append(log)
    if start_dev > START_TOL:
        raise SystemExit(f"control (3): the untrained interpreter is not its start formula ({start_dev:.2e}). Stopping.")
    epochs, mean = epoch_rule(logs)
    doc = {"start": start, "epochs": epochs, "formulas_validation": val, "logs": logs,
           "mean_by_epoch": mean, "control_3_max_deviation": start_dev, "halves": halves,
           "models": {n: {"revision": p["revision"], "sigma": p["sigma"], "controls": p["controls"],
                          "r": p["r"], "seconds_prepare": p["seconds_prepare"],
                          "formula_validation": {k: it["formula_r2"] for k, it in p["points"].items()},
                          "n_informative": {k: it["n_informative"] for k, it in p["points"].items()}}
                      for n, p in prepared.items()},
           "config": c, "environment": environment(), "wall_seconds": time.time() - t0}
    if out:
        with open(out, "w") as f:
            json.dump(doc, f, indent=1)
    print(f"pilot: start {start}, epochs {epochs}; control (3) {start_dev:.1e} ({time.time() - t0:.0f}s)", flush=True)
    return doc


def pilot_summary(doc: dict) -> dict:
    """From the pilot file: the choices by the rule, recomputed from its numbers."""
    val = doc["formulas_validation"]
    logs = [{int(ep): v for ep, v in lg.items()} for lg in doc["logs"]]
    epochs, mean = epoch_rule(logs)
    return {"start": start_rule(val), "epochs": epochs, "mean_by_epoch": mean, "formulas_validation": val}


'''


In [ ]:
#@title 2.20 Происхождение, прогоны, проверка
PARTS[('pythia_interpreter.py', 8)] = r'''# ------------------------------------------------------------------ provenance and the runs
def environment() -> dict:
    env = P.environment({"mode": "fp32"})
    env["script_sha256"] = P.sha256_file(os.path.abspath(__file__))
    env["pythia_address_sha256"] = P.sha256_file(os.path.abspath(P.__file__))
    return env


def load_r0(path: str = R0_FILE) -> tuple[dict, str]:
    if not os.path.exists(path) and os.path.exists(os.path.join("results", path)):
        path = os.path.join("results", path)
    sha = P.sha256_file(path)
    if sha != R0_SHA256:
        raise SystemExit(f"{path}: sha256 {sha[:12]} is not R.0's result file. Stopping.")
    return {r["model"]: r for r in json.load(open(path))["per_model"]}, sha


def fold_of(seed: int) -> int:
    return next(k for k, f in enumerate(FOLDS) if seed in f)


def reader_path(out_dir: str, k: int, s: int) -> str:
    return os.path.join(out_dir, f"reader_fold{k}_seed{s}.pt")


def save_reader(m: Reader, path: str, meta: dict) -> str:
    torch.save({"state": {k: v.cpu() for k, v in m.state_dict().items()}, "meta": meta}, path + ".tmp")
    os.replace(path + ".tmp", path)
    return P.sha256_file(path)


def load_reader(path: str, device, script_sha: str | None = None) -> Reader | None:
    if not os.path.exists(path):
        return None
    blob = torch.load(path, map_location="cpu", weights_only=False)
    if script_sha is not None and blob["meta"].get("script_sha256") != script_sha:
        return None
    m = Reader()
    m.load_state_dict(blob["state"])
    return m.to(device).eval()


def write_json(path: str, doc: dict, exclusive: bool = False) -> None:
    if exclusive:
        with open(path, "x") as f:
            json.dump(doc, f, indent=1)
        return
    with open(path + ".tmp", "w") as f:
        json.dump(doc, f, indent=1)
    os.replace(path + ".tmp", path)


def provenance(started: str, r0_sha: str, c: dict, extra: dict | None = None) -> dict:
    out = {"started_utc": started, "environment": environment(), "r0_file_sha256": r0_sha,
           "revisions": P.REVISIONS, "r_last": P.R_STAR, "r_arrival": P.R_ARRIVAL, "start": START,
           "epochs": EPOCHS, "folds": FOLDS, "config": c,
           "streams": {"pairs": P.PAIR_STREAM, "formula": P.FORMULA_STREAM, "probe": P.PROBE_STREAM,
                       "random": P.RANDOM_STREAM, "interpreter": INTERP_STREAM, "order": ORDER_STREAM,
                       "curve": CURVE_STREAM},
           "unit": "seed (1..9), mean over 14m-160m"}
    out.update(extra or {})
    return out


def main_run(device, c: dict, out_dir: str, started: str, models=None, epochs: int | None = None,
             out: str = OUT_MAIN) -> dict:
    """Prepare the 36 check models, train the nine interpreters (3 groups x 3 seeds), read every check
    model with its group's ensemble and write the confirmatory file."""
    t0 = time.time()
    r0, r0_sha = load_r0()
    os.makedirs(out_dir, exist_ok=True)
    me = P.sha256_file(os.path.abspath(__file__))
    models = models or [(P.check_name(size, seed), size, seed) for seed in P.CHECK_SEEDS for size in P.SIZES]
    prepared = {}
    for name, size, seed in models:
        prepared[name] = prepare(name, size, seed, device, c, splits=(0, 2), r0=r0.get(name), attr=True)
        pm = prepared[name]
        print(f"  {name}: prepared ({pm['seconds_prepare']:.0f}s), R.0 reproduced to "
              f"{pm.get('r0_check', {}).get('max_relative_deviation', float('nan')):.1e}", flush=True)
    readers, hashes = {}, {}
    for k, fold in enumerate(FOLDS):
        items = [it for pm in prepared.values() if pm["seed"] not in fold
                 for it in items_of(pm, START, device, 0)]
        for s in c["seeds"]:
            path = reader_path(out_dir, k, s)
            m = load_reader(path, device, me)
            if m is None:
                t1 = time.time()
                m = train_reader(items, (k, s), epochs or EPOCHS, device)
                hashes[f"fold{k}_seed{s}"] = save_reader(m, path, {"script_sha256": me, "fold": k, "seed": s,
                                                                   "epochs": epochs or EPOCHS,
                                                                   "training_items": len(items)})
                print(f"  group {k + 1} seed {s}: trained on {len(items)} items ({time.time() - t1:.0f}s)", flush=True)
            else:
                hashes[f"fold{k}_seed{s}"] = P.sha256_file(path)
                print(f"  group {k + 1} seed {s}: kept from {path}", flush=True)
            readers[(k, s)] = m
    records = []
    for name, pm in prepared.items():
        k = fold_of(pm["seed"]) if pm["seed"] in P.CHECK_SEEDS else 0
        rec = evaluate(pm, [readers[(k, s)] for s in c["seeds"]], START, device)
        rec["fold"] = k
        if name in r0:
            for key in ("last", "arrival", "attr"):
                rec[key]["das"] = r0[name][R0_KEY[key]]["das"]
        rec["environment"] = environment()
        write_json(os.path.join(out_dir, f"{name}.json"), rec)
        records.append(rec)
        print(f"  {name}: interpreter last {rec['last']['interpreter']:+.3f} arrival "
              f"{rec['arrival']['interpreter']:+.3f} attr {rec['attr']['interpreter']:+.3f}", flush=True)
    doc = {"provenance": provenance(started, r0_sha, c, {"reader_sha256": hashes}),
           "result": aggregate(records), "per_model": records, "wall_seconds": time.time() - t0}
    write_json(out, doc, exclusive=True)
    print("read-out:", doc["result"]["read_out"], "| rivals:", doc["result"]["rivals"], flush=True)
    print(f"wrote {out} ({time.time() - t0:.0f}s)", flush=True)
    return doc


def run_curve(device, c: dict, out_dir: str, started: str, models=None, out: str = OUT_CURVE) -> dict:
    """Descriptive: DAS(n) on the check models with the first n training pairs (R.0's recipe, the restart
    with the lowest training loss), test R^2; one file per model, so a broken session resumes."""
    t0 = time.time()
    r0, r0_sha = load_r0()
    os.makedirs(out_dir, exist_ok=True)
    me = P.sha256_file(os.path.abspath(__file__))
    models = models or [(P.check_name(size, seed), size, seed) for seed in P.CHECK_SEEDS for size in P.SIZES]
    recs = []
    for name, size, seed in models:
        path = os.path.join(out_dir, f"curve_{name}.json")
        if os.path.exists(path) and json.load(open(path))["environment"]["script_sha256"] == me:
            recs.append(json.load(open(path)))
            print(f"  curve {name}: already done, kept", flush=True)
            continue
        t1 = time.time()
        hf, model, vocab, rev = P.load(name, device)
        sigma = P.probe_sigma(model, vocab)
        rs = {"last": P.R_STAR[size], "arrival": P.R_ARRIVAL[size]}
        rec = {"model": name, "size": size, "seed": seed, "revision": rev, "sigma": sigma}
        for key in POINTS:
            if key == "arrival" and rs["arrival"] == rs["last"]:
                rec["arrival"] = dict(rec["last"], same_as_last=True)
                continue
            r = rs[key]
            test = pair_set(model, vocab, "SUBJ", r, sigma, 2, c["n_test"])
            row = {"r": r, "n_informative": test.informative()}
            for n in c["curve_n"]:
                train = pair_set(model, vocab, "SUBJ", r, sigma, 0, n)
                best = None
                for rs_ in range(c["curve_restarts"]):
                    g = torch.Generator().manual_seed(P.seed_of(CURVE_STREAM, P.SIZE_ID[size], seed,
                                                                POINTS.index(key), n, rs_))
                    q = P.das_steps(train, {"batch": PAIR_BATCH, "lr": 1e-2}, g, c["curve_steps"])
                    fit = train.r2(q)[0]
                    fit = -np.inf if not np.isfinite(fit) else fit
                    if best is None or fit > best[0]:
                        best = (fit, q)
                row[str(n)] = test.r2(best[1])[0]
            if name in r0:
                row["5120"] = r0[name][R0_KEY[key]]["das"]
            rec[key] = row
        rec["seconds"] = time.time() - t1
        rec["environment"] = environment()
        write_json(path, rec)
        recs.append(rec)
        del hf, model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"  curve {name}: " + " ".join(f"{key} " + " ".join(f"{n}:{rec[key][str(n)]:+.2f}" for n in c["curve_n"])
                                             for key in POINTS) + f" ({rec['seconds']:.0f}s)", flush=True)
    ns = [str(n) for n in c["curve_n"]] + (["5120"] if all("5120" in r["last"] for r in recs) else [])
    means = {}
    for key in POINTS:
        means[key] = {}
        for n in ns:
            vals = []
            for seed in P.CHECK_SEEDS:
                v = [r[key][n] for r in recs if r["seed"] == seed and r[key]["n_informative"] >= P.MIN_INFORMATIVE]
                vals.append(float(np.nanmean(v)) if v else float("nan"))
            means[key][n] = {"mean": float(np.nanmean(vals)), "ci95": P.boot(vals), "units": vals}
    worth = {}
    if os.path.exists(OUT_MAIN):
        main = json.load(open(OUT_MAIN))["result"]["means"]
        for key in POINTS:
            interp = main[key]["interpreter"]["mean"]
            worth[key] = {"interpreter": interp,
                          "smallest_n_with_das_at_least_the_interpreter": next(
                              (int(n) for n in ns if means[key][n]["mean"] >= interp), None)}
    doc = {"provenance": provenance(started, r0_sha, c), "means": means, "interpreter_worth": worth,
           "per_model": recs, "wall_seconds": time.time() - t0}
    if out:
        write_json(out, doc)
        print(f"wrote {out} ({time.time() - t0:.0f}s)", flush=True)
    return doc


def run_410(device, c: dict, out_dir: str, started: str, seeds=P.CHECK_SEEDS, out: str = OUT_410) -> dict:
    """Descriptive: 410m, both points, read by the ensemble of all nine interpreters, against the formulas."""
    t0 = time.time()
    me = P.sha256_file(os.path.abspath(__file__))
    readers = [load_reader(reader_path(out_dir, k, s), device, me) for k in range(len(FOLDS)) for s in c["seeds"]]
    if any(m is None for m in readers):
        raise SystemExit("the interpreters of the main run are not in this folder (or not of this script). Stopping.")
    recs = []
    for seed in seeds:
        name = P.check_name(P.SIZE_410, seed)
        path = os.path.join(out_dir, f"{name}.json")
        if os.path.exists(path) and json.load(open(path))["environment"]["script_sha256"] == me:
            recs.append(json.load(open(path)))
            print(f"  {name}: already done, kept", flush=True)
            continue
        pm = prepare(name, P.SIZE_410, seed, device, c, splits=(2,))
        rec = evaluate(pm, readers, START, device)
        rec["environment"] = environment()
        write_json(path, rec)
        recs.append(rec)
        del pm
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"  {name}: interpreter last {rec['last']['interpreter']:+.3f} arrival "
              f"{rec['arrival']['interpreter']:+.3f}", flush=True)
    arms = ("interpreter",) + UNLAB + LAB + ("random",)
    means = {key: {arm: {"mean": float(np.nanmean([r[key][arm] for r in recs])),
                         "ci95": P.boot([r[key][arm] for r in recs])} for arm in arms} for key in POINTS}
    doc = {"provenance": provenance(started, load_r0()[1], c), "means": means,
           "strongest_without_labels": {key: max(UNLAB, key=lambda a: means[key][a]["mean"]) for key in POINTS},
           "strongest_with_labels": {key: max(LAB, key=lambda a: means[key][a]["mean"]) for key in POINTS},
           "per_model": recs, "wall_seconds": time.time() - t0}
    if out:
        write_json(out, doc)
        print(f"wrote {out} ({time.time() - t0:.0f}s)", flush=True)
    return doc


def smoke(device) -> int:
    """Code path end to end on pythia-14m (a pilot model) with tiny sizes; the numbers mean nothing."""
    t0 = time.time()
    c = SMOKE
    os.makedirs("smoke_tmp", exist_ok=True)
    pm = prepare("pythia-14m", "14m", 0, device, c, splits=(0, 1, 2), attr=True)
    start = {"last": "gradient", "arrival": "gradient"}
    items = items_of(pm, start, device, 0)
    x = items[0]["x"]
    with torch.no_grad():
        q0 = Reader().to(device).eval()(x)
    dev0 = float((q0 - x["s"]).abs().max())
    m = train_reader(items, (0, 0), c["epochs"], device)
    rec = evaluate(pm, [m], start, device)
    rec["control_3_untrained_minus_start"] = dev0
    recs = [dict(rec, seed=s, size=z) for s in P.CHECK_SEEDS for z in P.SIZES]
    for r in recs:
        for key in ("last", "arrival", "attr"):
            r[key] = dict(r[key], das=float("nan"))
    agg = aggregate(recs)
    write_json("smoke_tmp/smoke.json", {"record": rec, "aggregate_on_copies": agg})
    run_curve(device, c, "smoke_tmp", "smoke", models=[("pythia-14m", "14m", 0)], out="")
    print(f"smoke done ({time.time() - t0:.0f}s): untrained = start to {dev0:.1e}; "
          f"interpreter last {rec['last']['interpreter']:+.3f}; nothing kept but smoke_tmp/", flush=True)
    return 0


def main(argv=None) -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--pilot", action="store_true")
    ap.add_argument("--pilot-summary", default=None, help="print the choices derived from a pilot file")
    ap.add_argument("--curve", action="store_true", help="descriptive: the DAS data curve")
    ap.add_argument("--m410", action="store_true", help="descriptive: transfer to 410m")
    ap.add_argument("--out", default="R1")
    ap.add_argument("--verify", default=None)
    a = ap.parse_args(argv)
    if a.verify:
        return verify(a.verify)
    if a.pilot_summary:
        ps = pilot_summary(json.load(open(a.pilot_summary)))
        print("formulas, mean validation R^2 over the pilot models:")
        for pt, v in ps["formulas_validation"].items():
            print(f"  {pt:8s}", " ".join(f"{k} {x:+.3f}" for k, x in v.items()))
        print("mean validation R^2 by epoch:", {e: round(x, 4) for e, x in ps["mean_by_epoch"].items()})
        print("by the rule: start", ps["start"], "epochs", ps["epochs"])
        same = ps["start"] == START and ps["epochs"] == EPOCHS
        print("the protocol's constants agree with this file:", same)
        return 0 if same else 5
    P.set_determinism()
    torch.set_num_threads(4)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    started = datetime.now(timezone.utc).isoformat(timespec="seconds")
    if a.smoke:
        return smoke(device)
    if a.pilot:
        pilot(device, FULL, OUT_PILOT)
        return 0
    if START is None or EPOCHS is None:
        print("the pilot's choices are not in the protocol yet. Stopping.")
        return 3
    if a.curve:
        run_curve(device, FULL, a.out, started)
        return 0
    if a.m410:
        run_410(device, FULL, a.out, started)
        return 0
    if os.path.exists(OUT_MAIN):
        print(f"{OUT_MAIN} exists; the confirmatory result is kept as it is")
        return 0
    main_run(device, FULL, a.out, started)
    return 0


def verify(path: str) -> int:
    """Locally, before a result is committed: every record of the file was computed by this script as it
    stands at the protocol commit, with pythia_address.py of that commit."""
    here = os.path.relpath(os.path.abspath(__file__))
    there = os.path.relpath(os.path.abspath(P.__file__))
    show = lambda rev, f: subprocess.run(["git", "show", f"{rev}:{f}"], capture_output=True, check=True).stdout
    commits = subprocess.run(["git", "log", "--format=%H %s", "--", here], capture_output=True, text=True,
                             check=True).stdout.splitlines()
    protocol = [line.split(" ", 1) for line in commits if line.split(" ", 1)[1].startswith(PROTOCOL_SUBJECT)]
    if len(protocol) != 1:
        print(f"{len(protocol)} protocol commits found for {here}. MISMATCH")
        return 4
    rev = protocol[0][0]
    mine = hashlib.sha256(show(rev, here)).hexdigest()
    theirs = hashlib.sha256(show(rev, there)).hexdigest()
    doc = json.load(open(path))
    envs = [doc["provenance"]["environment"]] + [r["environment"] for r in doc.get("per_model", [])]
    ok = all(e["script_sha256"] == mine and e["pythia_address_sha256"] == theirs for e in envs)
    ok &= doc["provenance"]["r0_file_sha256"] == R0_SHA256
    print(f"protocol commit {rev[:7]}: script {mine[:12]}, pythia_address {theirs[:12]}; "
          f"{len(envs)} records -> {'OK' if ok else 'MISMATCH'}")
    return 0 if ok else 4


if __name__ == "__main__":
    sys.exit(main())
'''


In [ ]:
#@title 2.21 Сборка двух файлов и проверка sha256 (обязательно)
import hashlib
EXPECTED = {"pythia_address.py": [11, "6c0606476a6e0c1f80ec6eec2b14bd073184e1eb2b3607c1e7c5492ec85dfc8d"], "pythia_interpreter.py": [9, "6923d17a479575296ace2bcfcd679cd46e0241f7a6fd397cfa028e75237ad107"]}
for fname, (count, sha) in EXPECTED.items():
    missing = [i for i in range(count) if (fname, i) not in PARTS]
    assert not missing, f"{fname}: не выполнены ячейки частей {missing}"
    data = "".join(PARTS[(fname, i)] for i in range(count)).encode("utf-8")
    got = hashlib.sha256(data).hexdigest()
    assert got == sha, f"{fname}: sha256 {got} не совпадает с {sha}: файл изменён, прогон недействителен"
    with open(fname, "wb") as f:
        f.write(data)
    print(fname, "записан; sha256 совпадает:", got[:16])


In [ ]:
#@title 2.22 Результат R.0 с Диска (для контроля 2; обязательно)
R0_ON_DRIVE = "/content/drive/MyDrive/R0/pythia_address.json"  #@param {type:"string"}
import hashlib, os, shutil
EXPECTED_R0 = "8c802368371d22115b9bf38d3d828e6c3d9bcfb95294afd76de1719145351a6e"
if not os.path.exists("pythia_address.json"):
    if os.path.exists(R0_ON_DRIVE):
        shutil.copy(R0_ON_DRIVE, "pythia_address.json")
    else:
        from google.colab import files
        print("на Диске нет", R0_ON_DRIVE, "- загрузите pythia_address.json (результат R.0)")
        for name, data in files.upload().items():
            open("pythia_address.json", "wb").write(data)
got = hashlib.sha256(open("pythia_address.json", "rb").read()).hexdigest()
assert got == EXPECTED_R0, f"pythia_address.json: sha256 {got[:16]} не тот, что назван в протоколе"
print("результат R.0 на месте; sha256 совпадает:", got[:16])


## 3. Пробный запуск

pythia-14m (пилотная модель) с крошечными настройками: загрузка, контроли, сигналы, обучение двух эпох, оценка,
сводка и кривая данных. Числа ничего не значат. Пишет только папку `smoke_tmp/`.

In [ ]:
#@title 3.1 Пробный запуск
RUN_SMOKE = False  #@param {type:"boolean"}
if RUN_SMOKE:
    get_ipython().system("python pythia_interpreter.py --smoke")
else:
    print("пропущено: пробный запуск уже сделан")


## 4. Пилот: старт и число эпох

Только 10 пилотных моделей (seed 0 каждого размера и шесть моделей 160m с отдельными seeds данных и весов);
проверочные модели не трогаются. Правило записано в протоколе до пилота: (1) старт каждой точки — формула без
меток с наибольшим средним R² на валидационных парах; (2) число эпох — по двум половинам пилотных моделей,
обучение на одной и валидация на другой. Пишет `pythia_interpreter_pilot.json`.

In [ ]:
#@title 4.1 Пилот
RUN_PILOT = False  #@param {type:"boolean"}
if RUN_PILOT:
    get_ipython().system("python pythia_interpreter.py --pilot")
else:
    print("пропущено: пилот уже сделан, его итог записан в протоколе")


In [ ]:
#@title 4.2 Итог пилота и скачивание файла
DOWNLOAD_PILOT = False  #@param {type:"boolean"}
import os
if os.path.exists("pythia_interpreter_pilot.json"):
    get_ipython().system("python pythia_interpreter.py --pilot-summary pythia_interpreter_pilot.json")
    if DOWNLOAD_PILOT:
        from google.colab import files
        files.download("pythia_interpreter_pilot.json")
else:
    print("файла пилота нет в этой папке")


## 5. Основной прогон

Только после того, как выбор пилота вписан в протокол и протокол закоммичен: до этого скрипт откажется работать.
36 проверочных моделей (seeds 1–9 × 14m, 31m, 70m, 160m): контроли R.0 и воспроизведение его чисел, сигналы,
пары; затем 9 интерпретаторов (3 группы × 3 seeds); затем чтение каждой модели ансамблем её группы. Пишет
`pythia_interpreter.json`, интерпретаторы — `R1/reader_*.pt`, модели — `R1/*.json`.

In [ ]:
#@title 5.1 Основной прогон
RUN_MAIN = True  #@param {type:"boolean"}
if RUN_MAIN:
    get_ipython().system("python pythia_interpreter.py --out R1")
else:
    print("пропущено: основной прогон запускается после пилота и коммита протокола")


## 6. Описательные части

410m (seeds 1–9, fp32) читается ансамблем всех девяти интерпретаторов основного прогона; затем кривая данных DAS
на проверочных моделях (16, 64, 256 и 1024 пары). Обе части продолжаются с места остановки.

In [ ]:
#@title 6.1 Перенос на 410m
RUN_410M = True  #@param {type:"boolean"}
if RUN_410M:
    get_ipython().system("python pythia_interpreter.py --m410 --out R1")
else:
    print("пропущено")


In [ ]:
#@title 6.2 Кривая данных DAS
RUN_CURVE = True  #@param {type:"boolean"}
if RUN_CURVE:
    get_ipython().system("python pythia_interpreter.py --curve --out R1")
else:
    print("пропущено")


## 7. Сводка и скачивание

In [ ]:
#@title 7.1 Итог и скачивание
import json, os
from google.colab import files
if os.path.exists("pythia_interpreter.json"):
    r = json.load(open("pythia_interpreter.json"))["result"]
    print("вывод по точкам:", r["read_out"])
    print("сильнейшие соперники:", r["rivals"])
    for key, arms in r["means"].items():
        print(f"{key:8s}", " ".join(f"{k} {x['mean']:+.3f}" for k, x in arms.items()))
for name in ("pythia_interpreter.json", "pythia_interpreter_410m.json", "pythia_interpreter_curve.json"):
    if os.path.exists(name):
        files.download(name)
    else:
        print(name, "ещё нет")
